# 02 — Contribution, league factors, replacement level (Phase 2)

**What?** One number per player-season that says how much he contributes on the pitch, with an
honest interval; a factor for what happens to that number when he changes league; and the level
a freely available player provides (so "surplus" = contribution above that level).

**Why?** This is the core of "same contribution for less money". The spec left several choices
open (what counts as contribution, how much history to use, how to build intervals), and each is
decided here by running the alternatives on the real data.

Inputs: the Phase 1 tables
(`scout.panel`). Each step ends with a "what we got" cell; the winner is then ported to
`scout.models` and re-run here to prove it reproduces the notebook.

## Contents

- Step 1 — Which numbers carry signal, per role, and how many minutes does a season need?
  - Step 1 — what we got
  - Step 1 check — the package reproduces the cells above
- Step 2 — What should "contribution" be made of?
  - Step 2 — what we got
- Step 3 — Does a player have his own "big-game" sensitivity?
  - Step 3 — what we got
- Step 4 — How much of a player's history should count?
- Step 5 — Is finishing a skill?
  - Step 4 — what we got
  - Step 5 — what we got
  - Step 4 check — the package reproduces the half-life 1.5 row
- Step 6 — Defenders, midfielders and keepers
  - Step 6 — what we got
  - Step 6 check — the package reproduces the keeper proxy
- Step 7 — What happens to a player's numbers when he changes league?
  - Step 7 — what we got
  - Step 7 check — the package reproduces the tier table
- Step 8 — What does a freely available player contribute?
  - Step 8 — what we got
  - Step 8 check — the package reproduces the p20 column
- Step 9 — How sure are we about a player's number?
  - Step 9 — what we got
  - Step 9 check — the package reproduces the outfield table, and the keeper proxy gets the same treatment
- Step 10 — Can the wider profile earn weights? (the "Rodri question")
  - Step 10 — what we got

*Steps 4 and 5 share one summary cell; Step 10 was run last — the owner's challenge to the finished framework.*


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 300)
from scout import config
from scout.data import understat
from scout.panel import market, player_match, stints, workrate

LEAGUE_TO_COMP = {league: comp for comp, league in config.BIG5.items()}

## Step 1 — Which numbers carry signal, per role, and how many minutes does a season need?

**What?** For every candidate per-90 quantity from the match data (non-penalty xG, xA, key
passes, shots, xG chain, xG build-up) and every work-rate quantity from Sofascore (tackles,
interceptions, …), how well does a player's number this season predict his number next season
(the correlation r), by role, at four minimum-minutes floors (300 / 600 / 900 / 1,200)?

**Why?** A quantity that does not repeat from one season to the next is noise and cannot describe
a player. The floor matters twice: too low and everything is noisy, too high and we throw away
the very players the backtest is about (players sold after a short season).

In [2]:
pm = player_match.build()
pm["competition_id"] = pm.league.map(LEAGUE_TO_COMP)
shots = understat.load("shots")

print("shots columns:", shots.columns.tolist())

# penalties: situation NA with xG 0.7612 (notebook 01, Part 4b); own goals are not the shooter's
is_pen = shots.situation.isna() & (shots.xg.round(4) == 0.7612)
pen_xg = shots[is_pen].groupby(["game_id", "player_id"]).xg.sum().rename("pen_xg")
pm = pm.merge(pen_xg, on=["game_id", "player_id"], how="left").fillna({"pen_xg": 0.0})
pm["npxg"] = pm.xg - pm.pen_xg

print(
    len(pm),
    "player-match rows | penalty xG subtracted from",
    int((pm.pen_xg > 0).sum()),
    "rows | own-goal rows in shots:",
    int((shots.result == "OwnGoal").sum()) if "result" in shots else "n/a",
)

QUANTITIES = ["npxg", "xa", "key_passes", "shots", "xg_chain", "xg_buildup", "goals", "assists"]
season_role = (
    pm.dropna(subset=["role"])
    .groupby(["competition_id", "season", "player_id", "role"])
    .agg(minutes=("minutes", "sum"), **{q: (q, "sum") for q in QUANTITIES})
    .reset_index()
)
per90 = season_role.copy()

for q in QUANTITIES:
    per90[q] = per90[q] / per90.minutes * 90

print(len(per90), "player-season-roles |", per90.groupby("role").size().to_dict())

shots columns: ['league', 'season', 'game', 'team', 'player', 'league_id', 'season_id', 'game_id', 'date', 'shot_id', 'team_id', 'player_id', 'assist_player_id', 'assist_player', 'xg', 'location_x', 'location_y', 'minute', 'body_part', 'situation', 'result']
630291 player-match rows | penalty xG subtracted from 1092 rows | own-goal rows in shots: 0
42157 player-season-roles | {'CB': 6770, 'CM': 8472, 'FB': 7317, 'GK': 2376, 'ST': 6179, 'W': 11043}


In [3]:
FLOORS = [300, 600, 900, 1200]


def year_to_year(frame, floor, quantities):
    kept = frame[frame.minutes >= floor]
    nxt = kept.assign(season=kept.season - 1)
    pairs = kept.merge(
        nxt, on=["competition_id", "player_id", "role", "season"], suffixes=("", "_next")
    )
    return pd.Series({q: pairs[q].corr(pairs[f"{q}_next"]) for q in quantities}), len(pairs)


rows = []

for role in ["GK", "CB", "FB", "CM", "W", "ST"]:
    for floor in FLOORS:
        r, n = year_to_year(per90[per90.role == role], floor, QUANTITIES)
        rows.append(pd.concat([pd.Series({"role": role, "floor": floor, "pairs": n}), r.round(2)]))

stability = pd.DataFrame(rows).set_index(["role", "floor"])

print(stability.to_string())
print("\nquantities with r >= 0.3 at 900, per role:")

for role, group in stability.xs(900, level="floor").iterrows():
    print(f"  {role}: {[q for q in QUANTITIES if group[q] >= 0.3]}")

            pairs  npxg    xa  key_passes  shots  xg_chain  xg_buildup  goals  assists
role floor                                                                            
GK   300     1020  0.06  0.04        0.12   0.02      0.56        0.56  -0.00     0.01
     600      890  0.15  0.07        0.11   0.05      0.65        0.65  -0.00     0.04
     900      800  0.15  0.06        0.12   0.08      0.64        0.64  -0.00     0.07
     1200     733  0.16  0.07        0.16   0.13      0.64        0.64  -0.00     0.08
CB   300     2966  0.27  0.21        0.40   0.46      0.67        0.67   0.13     0.06
     600     2492  0.30  0.24        0.45   0.52      0.71        0.71   0.16     0.08
     900     2047  0.33  0.28        0.49   0.55      0.73        0.72   0.15     0.10
     1200    1675  0.38  0.26        0.44   0.57      0.75        0.74   0.15     0.11
FB   300     2686  0.45  0.48        0.61   0.61      0.62        0.61   0.24     0.23
     600     2121  0.52  0.53        0.65  

In [4]:
# The cost side of the floor: paid departures from Big-5 clubs (backtest population) kept at each floor

moves = market.build()
paid = moves[(moves.kind == "paid")].copy()
paid["transfer_season"] = ("20" + paid.transfer_season.str[:2]).astype(int)
paid["prev_season"] = paid.transfer_season - (
    ~pd.to_datetime(paid.transfer_date).dt.month.isin([1, 2, 3])
).astype(int)
big5_clubs = stints.tm.load_player_club_seasons(list(config.BIG5), list(config.SEASONS))[
    ["club_id"]
].drop_duplicates()
paid = paid[paid.from_club_id.isin(big5_clubs.club_id) & paid.transfer_season.between(2015, 2024)]
last = stints.build(list(config.BIG5) + list(config.FEEDERS), list(config.SEASONS))[
    ["tm_player_id", "club_id", "season", "minutes"]
]
present = paid.merge(
    last,
    left_on=["player_id", "from_club_id", "prev_season"],
    right_on=["tm_player_id", "club_id", "season"],
).dropna(subset=["minutes"])

print(
    f"paid Big-5 departures 15/16 → 24/25 with a panel row at the selling club: {len(present)} (of {len(paid)})"
)

print(
    "kept at each floor:",
    {f: f"{(present.minutes >= f).mean():.1%}" for f in FLOORS},
    "| fee share kept:",
    {
        f: f"{present.transfer_fee[present.minutes >= f].sum() / present.transfer_fee.sum():.1%}"
        for f in FLOORS
    },
)

paid Big-5 departures 15/16 → 24/25 with a panel row at the selling club: 1706 (of 2736)
kept at each floor: {300: '85.2%', 600: '77.6%', 900: '69.6%', 1200: '60.7%'} | fee share kept: {300: '93.9%', 600: '89.4%', 900: '84.5%', 1200: '76.5%'}


**Work-rate quantities** (tackles, interceptions, recoveries, …) live in Sofascore, keyed by
Sofascore ids, while roles live in Understat. Joining them needs the Phase 1 identity twice
(Sofascore → Transfermarkt → Understat), which the next cell does before repeating the test.

In [5]:
from scout.data import reep, sofascore, transfermarkt as tm_loader
from scout.identity import build_team_lineage, load_overrides
from scout.panel import identity

comps = list(config.BIG5) + list(config.FEEDERS)
tm_panel = tm_loader.load_player_club_seasons(comps, list(config.SEASONS))
tm_clubs = tm_panel[["club_id", "club_name", "competition_id"]].drop_duplicates()
ss = sofascore.load()
us = understat.load("player_season")
us["competition_id"] = us.league.map(LEAGUE_TO_COMP)
lineage = build_team_lineage(
    tm_clubs,
    {
        "sofascore": ss[["competition_id", "team_name"]].drop_duplicates(),
        "understat": us[["competition_id", "team"]]
        .drop_duplicates()
        .rename(columns={"team": "team_name"}),
    },
    load_overrides("teams"),
)
people = reep.load_people()
tm_side = identity.transfermarkt_side(tm_panel)
ss_ids = identity.resolve_provider("sofascore", ss, tm_side, lineage, people).drop_duplicates(
    "provider_id"
)[["provider_id", "tm_player_id"]]
us_ids = identity.resolve_provider("understat", us, tm_side, lineage, people).drop_duplicates(
    "provider_id"
)[["provider_id", "tm_player_id"]]

print("ids:", len(ss_ids), "sofascore |", len(us_ids), "understat")

wr = pd.concat(
    [
        ss[["competition_id", "season", "sofascore_player_id", "minutesPlayed"]],
        workrate.sofascore_per90(ss),
    ],
    axis=1,
)
wr["tm_player_id"] = (
    wr.sofascore_player_id.astype(int).astype(str).map(ss_ids.set_index("provider_id").tm_player_id)
)
roles = per90[["competition_id", "season", "player_id", "role", "minutes"]].copy()
roles["tm_player_id"] = (
    roles.player_id.astype(int).astype(str).map(us_ids.set_index("provider_id").tm_player_id)
)
main_role = roles.sort_values("minutes", ascending=False).drop_duplicates(
    ["competition_id", "season", "tm_player_id"]
)
wr = wr.dropna(subset=["tm_player_id"]).merge(
    main_role[["competition_id", "season", "tm_player_id", "role"]],
    on=["competition_id", "season", "tm_player_id"],
)
wr["minutes"] = pd.to_numeric(wr.minutesPlayed)
wr["player_id"] = wr.tm_player_id
WR = [m for m in workrate.SHARED if m not in ("xg", "xa", "goals", "assists")] + [
    "possession_won_att_third_sofascore"
]

print(len(wr), "Sofascore player-seasons with a Big-5 role")
rows = []

for role in ["GK", "CB", "FB", "CM", "W", "ST"]:
    for floor in [300, 600, 900]:
        r, n = year_to_year(wr[wr.role == role], floor, WR)
        rows.append(pd.concat([pd.Series({"role": role, "floor": floor, "pairs": n}), r.round(2)]))

wr_stability = pd.DataFrame(rows).set_index(["role", "floor"])

print(
    wr_stability.rename(columns={"possession_won_att_third_sofascore": "poss_won_att3"}).to_string()
)

print("\nwork-rate quantities with r >= 0.3 at 600, per role:")

for role, group in wr_stability.xs(600, level="floor").iterrows():
    print(f"  {role}: {[q for q in WR if group[q] >= 0.3]}")

ids: 21090 sofascore | 9531 understat
26557 Sofascore player-seasons with a Big-5 role
            pairs  tackles  interceptions  recoveries  clearances  dribbles  key_passes  big_chances_created  accurate_passes  accurate_long_balls  fouls  saves  goals_conceded  poss_won_att3
role floor                                                                                                                                                                                     
GK   300      920     0.07           0.17        0.39        0.45      0.23        0.13                 0.06             0.71                 0.70   0.07   0.30            0.40          -0.01
     600      807     0.07           0.22        0.53        0.51      0.27        0.13                 0.09             0.75                 0.71   0.08   0.36            0.47          -0.01
     900      729     0.08           0.25        0.56        0.49      0.36        0.14                 0.06             0.78                 0.7

/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/mihailandreev/foot

### Step 1 — what we got

**Floor: 600 minutes** in a role-season. Going from 300 to 600 minutes buys +0.05 to +0.09 in r for
every outfield role; 600 to 900 buys only +0.03 to +0.05 while dropping eight more points of the
backtest population (77.6% of paid Big-5 departures kept at 600 and 89.4% of the fees, versus
69.6% / 84.5% at 900). Rejected: 300 (noisiest), 900 and 1,200 (less backtest for a smaller gain).

**Quantities that enter, per role** (r ≥ 0.3 at 600 minutes; goals and assists never enter — the
spec builds on expected quantities, and they are the least stable column in every role,
0.15–0.45):

| Role | From the match data | Work-rate (Sofascore / FotMob, per 90) |
|---|---|---|
| GK | xG chain, xG build-up (distribution only) | recoveries, clearances, accurate passes, long balls, saves (0.36), goals conceded (0.47 — really a team number; the keeper's own measure is Step 6) |
| CB | npxG (0.30), key passes, shots, xG chain, xG build-up — not xA (0.24) | tackles, interceptions, recoveries, clearances, dribbles, key passes, passes, long balls, fouls, possession won up front — not big chances created (0.17) |
| FB, CM, W, ST | npxG, xA, key passes, shots, xG chain, xG build-up | all twelve |

Two things to carry forward: goals conceded per 90 is stable for outfielders (0.34–0.43) because it
is the *team's* defence, so it belongs to a team measure, not a player list; and the most stable
columns everywhere are volume/style measures (accurate passes 0.8+, dribbles 0.7+) — they say what
a player *does*, not how much it is worth. Ported: `scout.models.quantities`.

### Step 1 check — the package reproduces the cells above

**What?** `scout.models.quantities` rebuilds the per-90 table and the striker row at 600 minutes;
the numbers must match.

In [6]:
from scout.models import quantities

packaged = quantities.season_role_per90(pm.drop(columns=["pen_xg", "npxg"]), shots)

print(len(packaged), "player-season-roles (cell above: 42,157) | floor:", quantities.MIN_MINUTES)
r, n = year_to_year(
    packaged[packaged.role == "ST"],
    600,
    ["npxg", "xa", "key_passes", "shots", "xg_chain", "xg_buildup"],
)

print("ST at 600 — pairs:", n, "(above: 1,408) |", r.round(2).to_dict())
print("(above: npxg 0.59, xa 0.45, key_passes 0.63, shots 0.63, xg_chain 0.60, xg_buildup 0.57)")

42157 player-season-roles (cell above: 42,157) | floor: 600
ST at 600 — pairs: 1408 (above: 1,408) | {'npxg': 0.59, 'xa': 0.45, 'key_passes': 0.63, 'shots': 0.63, 'xg_chain': 0.6, 'xg_buildup': 0.57}
(above: npxg 0.59, xa 0.45, key_passes 0.63, shots 0.63, xg_chain 0.60, xg_buildup 0.57)


## Step 2 — What should "contribution" be made of?

**What?** Four candidates per player-season, all per 90: (a) raw expected output = npxG + xA;
(b) team-share = the player's share of his team's attacking output while he is on the pitch;
(c) plus-minus = the team's xG difference while he is on the pitch; (d) share and plus-minus
together. Baseline: goals + assists.

**Why?** The spec left this open. Three tests, and the third decides: does the measure repeat
year to year better than goals + assists; does it track team quality (a warning sign); and —
the real test — which measure, taken at club A this season, best predicts the player's output
at a *different* club next season? A measure that stays with the player when he moves is the one
a scout can use.

In [7]:
from scout.panel import team_season

tm_long = team_season.team_match_long(understat.load("team_match"))
tm_long["competition_id"] = tm_long.league.map(LEAGUE_TO_COMP)
team_game = tm_long[
    ["competition_id", "season", "game_id", "team_id", "np_xg_for", "np_xg_against"]
]
rows = pm.merge(team_game, on=["competition_id", "season", "game_id", "team_id"], how="inner")
share = rows.minutes / 90
rows["on_pitch_np_xg_for"] = rows.np_xg_for * share
rows["on_pitch_np_xg_diff"] = (rows.np_xg_for - rows.np_xg_against) * share
KEYS = ["competition_id", "season", "player_id", "team_id", "role"]
stint = (
    rows.dropna(subset=["role"])
    .groupby(KEYS)
    .agg(
        minutes=("minutes", "sum"),
        npxg=("npxg", "sum"),
        xa=("xa", "sum"),
        goals=("goals", "sum"),
        assists=("assists", "sum"),
        team_for=("on_pitch_np_xg_for", "sum"),
        team_diff=("on_pitch_np_xg_diff", "sum"),
    )
    .reset_index()
)

stint = stint[stint.minutes >= quantities.MIN_MINUTES].copy()
per = 90 / stint.minutes
stint["raw"] = (stint.npxg + stint.xa) * per
stint["team_share"] = (stint.npxg + stint.xa) / stint.team_for
stint["plus_minus"] = stint.team_diff * per
stint["ga"] = (stint.goals + stint.assists) * per

print(len(stint), "club-season-roles at ≥600 min |", stint.role.value_counts().to_dict())
VARIANTS = ["raw", "team_share", "plus_minus", "ga"]

23086 club-season-roles at ≥600 min | {'CM': 5052, 'W': 4809, 'CB': 4579, 'FB': 4101, 'ST': 2949, 'GK': 1596}


In [8]:
# Kill check 1: year-to-year stability (same player, same role, consecutive seasons — any club) vs G+A


def stability_by_role(frame, cols):
    out = {}
    for role, group in frame.groupby("role"):
        season_level = (
            group.groupby(["competition_id", "season", "player_id", "role"])[cols + ["minutes"]]
            .agg({**{c: "mean" for c in cols}, "minutes": "sum"})
            .reset_index()
        )
        nxt = season_level.assign(season=season_level.season - 1)
        pairs = season_level.merge(
            nxt, on=["competition_id", "season", "player_id", "role"], suffixes=("", "_next")
        )
        out[role] = {c: round(pairs[c].corr(pairs[f"{c}_next"]), 2) for c in cols} | {
            "pairs": len(pairs)
        }
    return pd.DataFrame(out).T


print("year-to-year r by role:")
print(stability_by_role(stint, VARIANTS).to_string())

# Kill check 2: correlation with the team's season npxG difference (does the measure track team quality?)
team_diff_season = (
    tm_long.groupby(["competition_id", "season", "team_id"])
    .apply(lambda g: (g.np_xg_for - g.np_xg_against).mean())
    .rename("team_season_diff")
    .reset_index()
)
with_team = stint.merge(team_diff_season, on=["competition_id", "season", "team_id"])

print("\ncorrelation with team season npxG difference, attacking roles (W, ST):")
att = with_team[with_team.role.isin(["W", "ST"])]

print({v: round(att[v].corr(att.team_season_diff), 2) for v in VARIANTS})

year-to-year r by role:
     raw  team_share  plus_minus    ga   pairs
CB  0.32        0.27        0.69  0.15  2481.0
CM  0.69        0.62        0.71  0.50  2656.0
FB  0.60        0.51        0.68  0.37  2109.0
GK  0.08        0.11        0.67  0.06   889.0
ST  0.60        0.34        0.65  0.49  1391.0
W   0.64        0.43        0.68  0.45  2159.0

correlation with team season npxG difference, attacking roles (W, ST):
{'raw': np.float64(0.55), 'team_share': np.float64(0.0), 'plus_minus': np.float64(0.93), 'ga': np.float64(0.48)}


In [9]:
# Kill check 3 (decider): movers — same player, ≥600 min at club A in season s and at a different club B in s+1

nxt = stint.assign(season=stint.season - 1).rename(
    columns={"team_id": "team_next", "competition_id": "comp_next"}
)
movers = stint.merge(
    nxt[["comp_next", "season", "player_id", "role", "team_next", "raw", "ga"]].rename(
        columns={"raw": "raw_next", "ga": "ga_next"}
    ),
    on=["season", "player_id", "role"],
)
movers = movers[movers.team_next != movers.team_id]
stayers = stint.merge(
    nxt[["comp_next", "season", "player_id", "role", "team_next", "raw"]].rename(
        columns={"raw": "raw_next"}
    ),
    on=["season", "player_id", "role"],
)
stayers = stayers[stayers.team_next == stayers.team_id]

print(
    "movers:",
    len(movers),
    "| stayers:",
    len(stayers),
    "| movers by role:",
    movers.role.value_counts().to_dict(),
)


def predict_next(frame, variant, target="raw_next"):
    x, y = frame[variant], frame[target]
    slope, intercept = np.polyfit(x, y, 1)
    resid = y - (slope * x + intercept)
    return round(x.corr(y), 3), round(float(np.sqrt((resid**2).mean())), 3)


results = {}

for role_set, label in [
    (["W", "ST"], "attacking (W, ST)"),
    (["CB", "FB", "CM"], "non-attacking (CB, FB, CM)"),
]:
    m = movers[movers.role.isin(role_set)]
    s = stayers[stayers.role.isin(role_set)]
    results[label] = {"n_movers": len(m)}
    for v in VARIANTS:
        results[label][f"{v} r / rmse"] = predict_next(m, v)
    both = m.copy()
    both["both"] = (
        np.polyfit(np.c_[m.team_share, m.plus_minus].T.tolist()[0], m.raw_next, 1)[0] * 0
    )  # placeholder replaced below
    X = np.c_[np.ones(len(m)), m.team_share, m.plus_minus]
    beta, *_ = np.linalg.lstsq(X, m.raw_next, rcond=None)
    pred = X @ beta
    results[label]["both (share + plus-minus) r / rmse"] = (
        round(np.corrcoef(pred, m.raw_next)[0, 1], 3),
        round(float(np.sqrt(((m.raw_next - pred) ** 2).mean())), 3),
    )
    results[label]["stayers: raw r"] = predict_next(s, "raw")[0]

print(pd.DataFrame(results).to_string())

movers: 3079 | stayers: 9973 | movers by role: {'CM': 679, 'CB': 616, 'W': 565, 'ST': 522, 'FB': 493, 'GK': 204}
                                   attacking (W, ST) non-attacking (CB, FB, CM)
n_movers                                        1087                       1788
raw r / rmse                          (0.447, 0.179)             (0.561, 0.081)
team_share r / rmse                    (0.315, 0.19)             (0.521, 0.084)
plus_minus r / rmse                   (0.259, 0.194)             (0.144, 0.097)
ga r / rmse                           (0.375, 0.186)             (0.485, 0.086)
both (share + plus-minus) r / rmse    (0.416, 0.182)             (0.545, 0.082)
stayers: raw r                                 0.693                      0.714


### Step 2 — what we got

**Chosen: raw expected output per 90 — npxG + xA — with no team adjustment**, for every outfield
role. On 23,086 club-season-roles at 600+ minutes:

| | raw | team-share | plus-minus | both | goals+assists |
|---|---|---|---|---|---|
| repeats year to year, W / ST (r) | 0.64 / 0.60 | 0.43 / 0.34 | 0.68 / 0.65 | – | 0.45 / 0.49 |
| tracks the team's season xG difference (r) | 0.55 | 0.00 | 0.93 | – | 0.48 |
| predicts output at a different club next season, 1,087 attacking movers (r) | **0.447** | 0.315 | 0.259 | 0.416 | 0.375 |
| same, 1,788 defenders and midfielders | **0.561** | 0.521 | 0.144 | 0.545 | 0.485 |

- Raw beats goals + assists on stability in every role (from CB 0.32 vs 0.15 to CM 0.69 vs 0.50).
- Team-share is *less* stable than raw and predicts worse: dividing by the team's volume adds the
  team's noise and strips signal that travels with the player. Rejected.
- Plus-minus is the most stable column and correlates 0.93 with team quality: it measures the
  team, and it does not travel (0.26 / 0.14 on movers). Rejected — and the spec's "plus-minus is
  required for defenders" does not survive in this form; Step 6 tries a proper with/without design.
- "Both" is fitted on the same movers and still loses. Rejected.
- Players who stay put are predicted at r 0.69–0.71: changing club costs about a quarter of the
  predictability — the size of the effect Phases 3–4 have to explain.

Ported: `scout.models.contribution.expected_output`.

## Step 3 — Does a player have his own "big-game" sensitivity?

**What?** Every player-match row gets the opponent's ClubElo rating on the match date. For each
player-season, the slope of his per-match output on the opponent's strength (how much less he
produces per +100 Elo of opponent), shrunk toward the role average. Tests: does a player's slope
repeat year to year, and does correcting a mover's output for the opponents he faced predict his
output at the next club any better than the raw number?

**Why?** The idea that some players "farm stats against weak teams" came up at the very start of
the project. The spec's rule: if the slope does not repeat and does not help prediction, it is
dropped and reported as "scout intuition unsupported".

In [10]:
from scout.data import clubelo

ELO_COUNTRY = clubelo.ELO_COUNTRY
listings = pd.concat(
    [clubelo.list_clubs_on(f"{s}-07-01").assign(season=s) for s in config.SEASONS],
    ignore_index=True,
)
elo_names = (
    listings[listings.Country.isin(ELO_COUNTRY)]
    .assign(competition_id=lambda d: d.Country.map(ELO_COUNTRY))[["competition_id", "Club"]]
    .drop_duplicates()
    .rename(columns={"Club": "team_name"})
)
elo_lineage = build_team_lineage(tm_clubs, {"clubelo": elo_names}, load_overrides("teams")).dropna(
    subset=["club_id"]
)
histories = {name: clubelo.fetch_club(name) for name in elo_lineage.team_name.unique()}
print(len(histories), "club histories |", sum(h.empty for h in histories.values()), "empty")

# Understat team -> Transfermarkt club -> ClubElo name
# lineage built here so Step 3 needs nothing from later steps
us_teams = us[["competition_id", "team"]].drop_duplicates().rename(columns={"team": "team_name"})
us_lineage = build_team_lineage(tm_clubs, {"understat": us_teams}, load_overrides("teams"))[
    ["competition_id", "team_name", "club_id"]
].rename(columns={"team_name": "team"})
team_ids = (
    us[["competition_id", "team", "team_id"]]
    .drop_duplicates()
    .merge(us_lineage, on=["competition_id", "team"])
)
club_to_elo = elo_lineage.drop_duplicates("club_id").set_index("club_id").team_name
team_ids["elo_name"] = team_ids.club_id.map(club_to_elo)
print("Understat teams with a ClubElo club:", f"{team_ids.elo_name.notna().mean():.1%}")

# the opponent of each player-match row, and the opponent's Elo on that date
opp = team_game[["competition_id", "season", "game_id", "team_id"]].merge(
    team_game[["competition_id", "season", "game_id", "team_id"]].rename(
        columns={"team_id": "opp_id"}
    ),
    on=["competition_id", "season", "game_id"],
)
opp = opp[opp.team_id != opp.opp_id].merge(
    team_ids[["competition_id", "team_id", "elo_name"]].rename(
        columns={"team_id": "opp_id", "elo_name": "opp_elo_name"}
    ),
    on=["competition_id", "opp_id"],
    how="left",
)
dates = tm_long[
    ["competition_id", "season", "game_id", "date"]
].drop_duplicates()  # player_match has no date column; tm_long does
opp = opp.merge(dates, on=["competition_id", "season", "game_id"])


def elo_lookup(frame):
    out = np.full(len(frame), np.nan)
    for name, idx in frame.groupby("opp_elo_name").indices.items():
        h = histories.get(name)
        if h is None or h.empty:
            continue
        d = pd.to_datetime(frame.date.iloc[idx]).dt.normalize().to_numpy()
        starts, ends, elos = h.From.to_numpy(), h.To.to_numpy(), h.Elo.to_numpy()
        pos = np.searchsorted(starts, d, side="right") - 1
        ok = (pos >= 0) & (d <= ends[np.clip(pos, 0, len(ends) - 1)])
        out[idx[ok]] = elos[pos[ok]]
    return out


opp["opp_elo"] = elo_lookup(opp)
coverage = (
    opp.groupby(["competition_id", "season"])
    .opp_elo.apply(lambda c: c.notna().mean())
    .unstack("season")
    .round(3)
)
print("share of team-matches with an opponent Elo on the match date:")
print(coverage.to_string())

340 club histories | 0 empty
Understat teams with a ClubElo club: 100.0%
share of team-matches with an opponent Elo on the match date:
season          2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
competition_id                                                                        
ES1              1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0
FR1              1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0
GB1              1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0
IT1              1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0
L1               1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0


In [11]:
# per-match expected output vs opponent Elo (centred within league-season), per player-season-role
pm_elo = (
    pm.dropna(subset=["role"])
    .merge(
        opp[["competition_id", "season", "game_id", "team_id", "opp_elo"]],
        on=["competition_id", "season", "game_id", "team_id"],
    )
    .dropna(subset=["opp_elo"])
)
pm_elo["out90"] = (pm_elo.npxg + pm_elo.xa) / pm_elo.minutes * 90
pm_elo["elo_c"] = (
    pm_elo.opp_elo - pm_elo.groupby(["competition_id", "season"]).opp_elo.transform("mean")
) / 100  # per 100 Elo points
pm_elo = pm_elo[pm_elo.minutes >= 30]
keys = ["competition_id", "season", "player_id", "role"]


def slope(g):
    if len(g) < 8:
        return pd.Series({"slope": np.nan, "se": np.nan, "n": len(g)})
    w = g.minutes.to_numpy()
    x = g.elo_c.to_numpy()
    y = g.out90.to_numpy()
    xm, ym = np.average(x, weights=w), np.average(y, weights=w)
    sxx = np.sum(w * (x - xm) ** 2)
    if sxx == 0:
        return pd.Series({"slope": np.nan, "se": np.nan, "n": len(g)})
    b = np.sum(w * (x - xm) * (y - ym)) / sxx
    resid = y - (ym + b * (x - xm))
    se = np.sqrt(np.sum(w * resid**2) / (w.sum() - 2) / sxx * w.mean())
    return pd.Series({"slope": b, "se": se, "n": len(g)})


slopes = pm_elo.groupby(keys).apply(slope).reset_index().dropna(subset=["slope"])
slopes = slopes.merge(per90[keys + ["minutes"]], on=keys)
slopes = slopes[slopes.minutes >= quantities.MIN_MINUTES]
print(
    len(slopes),
    "player-season-roles with a slope | mean slope by role (output change per +100 opponent Elo):",
)
print(slopes.groupby("role").slope.agg(["mean", "median", "std"]).round(4).to_string())
# shrink toward the role mean: tau^2 from the y2y covariance, noise = se^2
nxt = slopes.assign(season=slopes.season - 1)[keys + ["slope"]].rename(columns={"slope": "next"})
sp = slopes.merge(nxt, on=keys)
out = {}
for role, g in sp.groupby("role"):
    mu, tau2 = g.slope.mean(), max(float(np.cov(g.slope, g.next)[0, 1]), 1e-8)
    k = tau2 / (tau2 + g.se**2)
    shrunk = mu + k * (g.slope - mu)
    out[role] = {
        "pairs": len(g),
        "raw y2y r": round(g.slope.corr(g.next), 3),
        "shrunk vs next r": round(shrunk.corr(g.next), 3),
        "tau": round(np.sqrt(tau2), 4),
        "median k": round(float(k.median()), 2),
        "median se": round(float(g.se.median()), 4),
    }
print("\npersistence of the opponent slope:")
print(pd.DataFrame(out).T.to_string())

22249 player-season-roles with a slope | mean slope by role (output change per +100 opponent Elo):
        mean  median     std
role                        
CB   -0.0093 -0.0075  0.0367
CM   -0.0200 -0.0153  0.0633
FB   -0.0189 -0.0141  0.0605
GK   -0.0001  0.0000  0.0040
ST   -0.0619 -0.0613  0.1343
W    -0.0454 -0.0442  0.1117

persistence of the opponent slope:
     pairs  raw y2y r  shrunk vs next r     tau  median k  median se
CB  2410.0      0.011            -0.001  0.0036      0.02     0.0246
CM  2586.0      0.105             0.088  0.0192      0.22     0.0365
FB  2026.0      0.089             0.088  0.0167      0.18     0.0354
GK   860.0     -0.001            -0.002  0.0001      1.00     0.0000
ST  1323.0      0.044             0.084  0.0272      0.08     0.0909
W   2067.0      0.045             0.039  0.0238      0.09     0.0748


In [12]:
# does an opponent-adjusted output predict a mover's next-club output better than raw? (Step 2 design)
# adjusted = output the player would have at a league-average opponent: raw - slope_shrunk * mean(elo_c faced)
faced = (
    pm_elo.groupby(keys)
    .apply(lambda g: np.average(g.elo_c, weights=g.minutes))
    .rename("elo_faced")
    .reset_index()
)
adj = slopes.merge(faced, on=keys)
tau_by_role = {role: (v["tau"] ** 2) for role, v in out.items()}
mu_by_role = slopes.groupby("role").slope.mean()
adj["k"] = adj.apply(lambda r: tau_by_role[r.role] / (tau_by_role[r.role] + r.se**2), axis=1)
adj["slope_shrunk"] = adj.role.map(mu_by_role) + adj.k * (adj.slope - adj.role.map(mu_by_role))
base = stint.merge(adj[keys + ["slope_shrunk", "elo_faced"]], on=keys)
base["raw_adj"] = base.raw - base.slope_shrunk * base.elo_faced
nxt = base.assign(season=base.season - 1).rename(
    columns={"team_id": "team_next", "raw": "raw_next"}
)[["season", "player_id", "role", "team_next", "raw_next"]]
mv3 = base.merge(nxt, on=["season", "player_id", "role"])
mv3 = mv3[mv3.team_next != mv3.team_id]
for label, roles in [
    ("attacking (W, ST)", ["W", "ST"]),
    ("non-attacking (CB, FB, CM)", ["CB", "FB", "CM"]),
]:
    m = mv3[mv3.role.isin(roles)]
    print(
        f"{label}: movers n={len(m)} | raw r {m.raw.corr(m.raw_next):.3f} | opponent-adjusted r {m.raw_adj.corr(m.raw_next):.3f} | mean |elo_faced| {m.elo_faced.abs().mean():.2f} (hundreds of Elo)"
    )

attacking (W, ST): movers n=1009 | raw r 0.462 | opponent-adjusted r 0.462 | mean |elo_faced| 0.17 (hundreds of Elo)
non-attacking (CB, FB, CM): movers n=1728 | raw r 0.575 | opponent-adjusted r 0.575 | mean |elo_faced| 0.15 (hundreds of Elo)


### Step 3 — what we got

**Dropped; reported as "scout intuition unsupported".** Opponent Elo on the match date exists for
100% of Big-5 team-matches in every season (this also closes the last Phase 1 bar). The *average*
effect is real and in the expected direction: per +100 Elo of opponent, strikers produce 0.062
less xG + xA per 90, wingers 0.045, central midfielders 0.020, full-backs 0.019, centre-backs
0.009. The *player-specific* slope is not measurable at season level: it repeats year to year with
r 0.01 (CB) to 0.10 (CM), shrinkage keeps only 2–22% of a player's own estimate, and correcting a
mover's output for the opponents he faced leaves next-club prediction unchanged (r 0.462 → 0.462
attacking, 0.575 → 0.575 defenders and midfielders). Two reasons it cannot matter here: within a
league-season everyone faces almost the same schedule (mean deviation 15–17 Elo points, so the
average slope moves output by about 0.01 per 90 at most), and one season's per-match noise
(standard error 0.04–0.09) is larger than the true spread of slopes (0.004–0.027). Rejected: a
per-player slope in the contribution. Kept: the Elo join (`scout.panel.elo`) and the role-average
slope as a descriptive schedule effect; the writeup shows this as a negative result.

## Step 4 — How much of a player's history should count?

**What?** Candidates for turning several seasons into one number: last season only; the plain
average of the last three; a weighted average that halves the weight every 1, 1.5, 2 or 3
seasons. Test set: every player with 600+ minutes in four consecutive seasons; target = the
fourth season's output.

**Why?** One season is noisy; ten-year-old seasons are stale. The data decides where the balance
lies.

In [13]:
season_level = per90.copy()
season_level["expected_output"] = season_level.npxg + season_level.xa
season_level = season_level[season_level.minutes >= quantities.MIN_MINUTES][
    ["competition_id", "season", "player_id", "role", "expected_output", "minutes"]
]

# one row per player-role-season (a mover across leagues in one season keeps the bigger stint)
season_level = season_level.sort_values("minutes", ascending=False).drop_duplicates(
    ["season", "player_id", "role"]
)
wide = season_level.pivot(index=["player_id", "role"], columns="season", values="expected_output")
cases = []

for s in range(2017, 2026):
    block = wide[[s - 3, s - 2, s - 1, s]].dropna()
    cases.append(block.set_axis(["lag3", "lag2", "lag1", "target"], axis=1).assign(season=s))

cases = pd.concat(cases).reset_index()

print(
    len(cases),
    "player-role cases with four consecutive ≥600-minute seasons |",
    cases.role.value_counts().to_dict(),
)


def weighted(frame, half_life):
    w = np.array([0.5 ** (k / half_life) for k in (2, 1, 0)])  # lag3, lag2, lag1
    return (frame[["lag3", "lag2", "lag1"]].to_numpy() * w).sum(axis=1) / w.sum()


schemes = {
    "last season only": cases.lag1,
    "mean of three": cases[["lag3", "lag2", "lag1"]].mean(axis=1),
    "half-life 1": weighted(cases, 1.0),
    "half-life 1.5": weighted(cases, 1.5),
    "half-life 2": weighted(cases, 2.0),
    "half-life 3": weighted(cases, 3.0),
}

table = {}

for name, pred in schemes.items():
    pred = pd.Series(np.asarray(pred), index=cases.index)
    table[name] = {
        "r": round(pred.corr(cases.target), 3),
        "rmse": round(float(np.sqrt(((cases.target - pred) ** 2).mean())), 4),
    }
    for role in ["W", "ST", "CM"]:
        m = cases.role == role
        table[name][f"r {role}"] = round(pred[m].corr(cases.target[m]), 3)

print(pd.DataFrame(table).T.to_string())

5228 player-role cases with four consecutive ≥600-minute seasons | {'CM': 1256, 'CB': 1192, 'FB': 910, 'W': 818, 'ST': 618, 'GK': 434}
                      r    rmse    r W   r ST   r CM
last season only  0.876  0.1198  0.615  0.618  0.694
mean of three     0.897  0.1074  0.670  0.631  0.717
half-life 1       0.898  0.1069  0.670  0.650  0.732
half-life 1.5     0.899  0.1063  0.674  0.648  0.731
half-life 2       0.899  0.1063  0.674  0.645  0.729
half-life 3       0.899  0.1065  0.674  0.642  0.726


## Step 5 — Is finishing a skill?

**What?** Goals minus xG per 90 (both include penalties, so this is finishing only): does a
player's over- or under-performance repeat from one season to the next, and does last season's
number improve the prediction of next season's goals beyond xG?

**Why?** The spec expected this to fail, and says to report it either way: if finishing does not
repeat, contribution stays on expected quantities and the residual is stored as a description only.

In [14]:
xg_totals = (
    pm.dropna(subset=["role"])
    .groupby(["competition_id", "season", "player_id", "role"])
    .agg(xg_total=("xg", "sum"))
    .reset_index()
)
fin = per90[per90.minutes >= quantities.MIN_MINUTES].merge(
    xg_totals, on=["competition_id", "season", "player_id", "role"]
)
fin["xg_per90"] = fin.xg_total / fin.minutes * 90
fin["residual"] = fin.goals - fin.xg_per90

print("finishing residual by role — mean, sd:")
print(fin.groupby("role").residual.agg(["mean", "std", "size"]).round(3).to_string())
nxt = fin.assign(season=fin.season - 1)[
    ["competition_id", "season", "player_id", "role", "residual", "goals", "xg_per90"]
].rename(columns={"residual": "residual_next", "goals": "goals_next", "xg_per90": "xg_next"})
pairs = fin.merge(nxt, on=["competition_id", "season", "player_id", "role"])

print(
    "\nyear-to-year r of the residual:",
    pairs.groupby("role").apply(lambda g: round(g.residual.corr(g.residual_next), 3)).to_dict(),
)

for role in ["ST", "W", "CM"]:
    g = pairs[pairs.role == role]
    X1 = np.c_[np.ones(len(g)), g.xg_next]
    X2 = np.c_[np.ones(len(g)), g.xg_next, g.residual]
    r1 = np.corrcoef(X1 @ np.linalg.lstsq(X1, g.goals_next, rcond=None)[0], g.goals_next)[0, 1]
    r2 = np.corrcoef(X2 @ np.linalg.lstsq(X2, g.goals_next, rcond=None)[0], g.goals_next)[0, 1]
    print(
        f"{role}: next-season goals/90 predicted from next-season xG alone r={r1:.3f}; adding last season's residual r={r2:.3f} (n={len(g)})"
    )

finishing residual by role — mean, sd:
       mean    std  size
role                    
CB   -0.008  0.042  4565
CM   -0.004  0.065  5045
FB   -0.005  0.046  4098
GK     -0.0  0.003  1592
ST   -0.027  0.139  2941
W    -0.004  0.109  4829

year-to-year r of the residual: {'CB': 0.026, 'CM': 0.018, 'FB': 0.015, 'GK': 0.06, 'ST': 0.089, 'W': 0.068}
ST: next-season goals/90 predicted from next-season xG alone r=0.799; adding last season's residual r=0.801 (n=1408)
W: next-season goals/90 predicted from next-season xG alone r=0.764; adding last season's residual r=0.766 (n=2184)
CM: next-season goals/90 predicted from next-season xG alone r=0.770; adding last season's residual r=0.770 (n=2670)


### Step 4 — what we got

5,228 player-role cases with four consecutive 600-minute seasons. Last season alone predicts the
next at r 0.876 (error 0.120); pooling three seasons lifts every scheme to 0.897–0.899 (within a
role: W 0.67, ST 0.65, CM 0.73 — the pooled r is inflated by differences between roles).
Half-lives of 1 to 3 seasons differ by 0.001 in r; **half-life 1.5 has the lowest error (0.1063)**
and is kept. Rejected: last season only (clearly worse), the plain average (same r, error 0.1074).
Ported: `scout.models.recency` — missing seasons re-weight over what exists.

### Step 5 — what we got

Finishing does not repeat: year-to-year r is 0.09 for strikers, 0.07 for wingers, 0.02 for the
rest, and adding last season's residual to next season's xG changes the prediction of next
season's goals by at most +0.002 in r (ST 0.799 → 0.801). As the spec expected. Contribution stays
on expected quantities; the residual is stored as its own descriptive column. Side note for the
writeup: strikers score 0.027 goals per 90 fewer than their xG on average (sd 0.14), other roles
within 0.01 — Understat's xG is well calibrated by role.

### Step 4 check — the package reproduces the half-life 1.5 row

**What?** `scout.models.recency` applied to the same cases; r and error must match.

In [15]:
from scout.models import recency

pred = recency.weighted_history(cases[["lag1", "lag2", "lag3"]])

print(
    "half-life",
    recency.HALF_LIFE,
    "| r:",
    round(pred.corr(cases.target), 3),
    "| rmse:",
    round(float(np.sqrt(((cases.target - pred) ** 2).mean())), 4),
    "(above: 0.899 / 0.1063)",
)

half-life 1.5 | r: 0.899 | rmse: 0.1063 (above: 0.899 / 0.1063)


## Step 6 — Defenders, midfielders and keepers

**What?** *Defensive value, properly measured:* for each player-club-season, the team's non-penalty
xG conceded per 90 in the matches he played (45+ minutes) versus the matches he missed entirely,
at least 5 of each — "on/off". The same for xG created. Tests as in Step 2: does it repeat year to
year, and does it travel with a mover? *Keepers:* three candidate measures — the Understat proxy
(xG of on-target shots faced minus goals conceded, per 90), FotMob's "goals prevented", and
Sofascore saves per 90. Bar: repeats year to year with r ≥ 0.3 (Phase 0 found 0.4–0.5).

**Why?** Expected output (npxG + xA) says nothing about defending. The spec wanted plus-minus for
defenders; Step 2 showed the simple version measures the team, so this is the proper with/without
design. Keepers need their own measure entirely.

In [16]:
# on/off: every (club, season, match) the club played, joined to the player's minutes in it

club_matches = team_game.rename(columns={"team_id": "club_team_id"})
appearances = pm.dropna(subset=["role"])[
    ["competition_id", "season", "game_id", "team_id", "player_id", "role", "minutes"]
]
player_clubs = (
    appearances.groupby(["competition_id", "season", "team_id", "player_id"])
    .agg(role=("role", lambda r: r.value_counts().index[0]), minutes=("minutes", "sum"))
    .reset_index()
)
player_clubs = player_clubs[player_clubs.minutes >= quantities.MIN_MINUTES]
grid = player_clubs.merge(
    club_matches,
    left_on=["competition_id", "season", "team_id"],
    right_on=["competition_id", "season", "club_team_id"],
)
grid = grid.merge(
    appearances[["competition_id", "season", "game_id", "team_id", "player_id", "minutes"]].rename(
        columns={"minutes": "mins_in_match"}
    ),
    on=["competition_id", "season", "game_id", "team_id", "player_id"],
    how="left",
).fillna({"mins_in_match": 0})
grid["state"] = np.select(
    [grid.mins_in_match >= 45, grid.mins_in_match == 0], ["on", "off"], "partial"
)
agg = (
    grid[grid.state != "partial"]
    .groupby(["competition_id", "season", "team_id", "player_id", "role", "state"])
    .agg(matches=("game_id", "size"), xga=("np_xg_against", "mean"), xgf=("np_xg_for", "mean"))
    .unstack("state")
)
agg.columns = [f"{a}_{b}" for a, b in agg.columns]
onoff = agg[(agg.matches_on >= 5) & (agg.matches_off >= 5)].reset_index()
onoff["on_off_xga"] = onoff.xga_off - onoff.xga_on
onoff["on_off_xgf"] = onoff.xgf_on - onoff.xgf_off

print(
    len(onoff),
    "player-club-seasons with ≥5 matches on and off |",
    onoff.role.value_counts().to_dict(),
)

print("on/off xGA by role — mean, sd:")
print(onoff.groupby("role").on_off_xga.agg(["mean", "std"]).round(3).to_string())

17327 player-club-seasons with ≥5 matches on and off | {'CM': 3788, 'CB': 3758, 'W': 3410, 'FB': 3350, 'ST': 2058, 'GK': 963}
on/off xGA by role — mean, sd:
       mean    std
role              
CB    0.006  0.317
CM    0.002   0.32
FB   -0.002  0.315
GK     0.02  0.301
ST    0.034  0.331
W     0.011  0.334


In [17]:
def onoff_stability(frame, col):
    season_level = frame.sort_values("matches_on", ascending=False).drop_duplicates(
        ["season", "player_id", "role"]
    )
    nxt = season_level.assign(season=season_level.season - 1)
    pairs = season_level.merge(nxt, on=["season", "player_id", "role"], suffixes=("", "_next"))
    stay = pairs[pairs.team_id == pairs.team_id_next]
    move = pairs[pairs.team_id != pairs.team_id_next]
    return pd.Series(
        {
            "stayers r": round(stay[col].corr(stay[f"{col}_next"]), 3),
            "n stay": len(stay),
            "movers r": round(move[col].corr(move[f"{col}_next"]), 3),
            "n move": len(move),
        }
    )


print("on/off xGA — year-to-year, same club vs after a move:")
print(
    pd.DataFrame(
        {
            role: onoff_stability(onoff[onoff.role == role], "on_off_xga")
            for role in ["GK", "CB", "FB", "CM", "W", "ST"]
        }
    ).T.to_string()
)

print("\non/off xG for:")
print(
    pd.DataFrame(
        {
            role: onoff_stability(onoff[onoff.role == role], "on_off_xgf")
            for role in ["CB", "FB", "CM", "W", "ST"]
        }
    ).T.to_string()
)

print(
    "\nfor comparison, raw expected output on the same movers/stayers definition: attacking movers r 0.447, stayers 0.69 (Step 2)"
)

on/off xGA — year-to-year, same club vs after a move:
    stayers r  n stay  movers r  n move
GK      0.019   229.0     0.092    94.0
CB      0.078  1415.0    -0.038   424.0
FB     -0.025  1183.0    -0.038   345.0
CM      0.061  1197.0     0.075   423.0
W      -0.018   812.0     0.044   321.0
ST      0.076   440.0     0.050   283.0

on/off xG for:
    stayers r  n stay  movers r  n move
CB     -0.009  1415.0    -0.050   424.0
FB      0.003  1183.0     0.112   345.0
CM      0.030  1197.0     0.078   423.0
W      -0.004   812.0    -0.052   321.0
ST      0.080   440.0     0.110   283.0

for comparison, raw expected output on the same movers/stayers definition: attacking movers r 0.447, stayers 0.69 (Step 2)


In [18]:
# Keeper proxy from the shots table: on-target xG faced minus goals conceded, per 90 of the keeper's minutes

on_target = shots[shots.result.isin(["Goal", "Saved Shot"])].copy()
on_target["competition_id"] = on_target.league.map(LEAGUE_TO_COMP)
faced = (
    on_target.groupby(["competition_id", "season", "game_id", "team_id"])
    .agg(xg_on_target=("xg", "sum"), goals=("result", lambda r: (r == "Goal").sum()))
    .reset_index()
)
keepers = pm[(pm.role == "GK") & (pm.minutes >= 45)][
    ["competition_id", "season", "game_id", "team_id", "player_id", "minutes"]
]

# the shots a keeper faces are the *other* team's shots in his game
opponent = team_game[["competition_id", "season", "game_id", "team_id"]].merge(
    team_game[["competition_id", "season", "game_id", "team_id"]].rename(
        columns={"team_id": "opp_id"}
    ),
    on=["competition_id", "season", "game_id"],
)
opponent = opponent[opponent.team_id != opponent.opp_id]
kp = (
    keepers.merge(opponent, on=["competition_id", "season", "game_id", "team_id"])
    .merge(
        faced.rename(columns={"team_id": "opp_id"}),
        on=["competition_id", "season", "game_id", "opp_id"],
        how="left",
    )
    .fillna({"xg_on_target": 0, "goals": 0})
)
kp_season = (
    kp.groupby(["competition_id", "season", "player_id"])
    .agg(
        minutes=("minutes", "sum"),
        matches=("game_id", "size"),
        xg_on_target=("xg_on_target", "sum"),
        goals=("goals", "sum"),
    )
    .reset_index()
)
kp_season = kp_season[kp_season.minutes >= quantities.MIN_MINUTES]
kp_season["prevented_per90"] = (kp_season.xg_on_target - kp_season.goals) / kp_season.minutes * 90
nxt = kp_season.assign(season=kp_season.season - 1)[
    ["competition_id", "season", "player_id", "prevented_per90"]
].rename(columns={"prevented_per90": "next"})
pairs = kp_season.merge(nxt, on=["competition_id", "season", "player_id"])

print(
    len(kp_season),
    "keeper-seasons ≥600 min | proxy mean",
    round(kp_season.prevented_per90.mean(), 3),
    "sd",
    round(kp_season.prevented_per90.std(), 3),
)

print(
    f"Understat proxy year-to-year r = {pairs.prevented_per90.corr(pairs.next):.3f} (n={len(pairs)}; Phase 0 on 3 PL seasons: 0.42–0.52)"
)

# FotMob goals prevented and Sofascore saves, for keepers with an identity
from scout.data import fotmob

fm = fotmob.load()
gp = fm[fm.stat == "_goals_prevented"][
    ["competition_id", "season", "fotmob_player_id", "stat_value", "minutes_played"]
].rename(columns={"stat_value": "goals_prevented"})
gp = gp[gp.minutes_played >= quantities.MIN_MINUTES]
gp_next = gp.assign(season=gp.season - 1)[
    ["competition_id", "season", "fotmob_player_id", "goals_prevented"]
].rename(columns={"goals_prevented": "next"})
gpp = gp.merge(gp_next, on=["competition_id", "season", "fotmob_player_id"])

print(
    f"FotMob goals prevented year-to-year r = {gpp.goals_prevented.corr(gpp.next):.3f} (n={len(gpp)}, all leagues on disk)"
)
gk_wr = wr[wr.role == "GK"]

print(
    "Sofascore saves per 90 year-to-year r at 600 (Step 1):", wr_stability.loc[("GK", 600), "saves"]
)

1591 keeper-seasons ≥600 min | proxy mean -0.565 sd 0.248
Understat proxy year-to-year r = 0.336 (n=889; Phase 0 on 3 PL seasons: 0.42–0.52)


FotMob goals prevented year-to-year r = 0.362 (n=572, all leagues on disk)
Sofascore saves per 90 year-to-year r at 600 (Step 1): 0.36


**Two follow-ups.** Keepers: do the three measures agree, and is their average more stable than any
one of them? Outfield defensive *actions* (tackles, interceptions, …; Step 1 found them stable
within a club): do they travel with a mover, which the on/off numbers did not?

In [19]:
# keepers: join the Understat proxy to FotMob goals prevented and Sofascore saves via Transfermarkt ids

fm_ids = identity.resolve_provider(
    "fotmob",
    fm[fm.stat == "mins_played"]
    .rename(columns={"stat_value": "fm_minutes"})[
        ["competition_id", "season", "fotmob_player_id", "player_name", "team_name", "fm_minutes"]
    ]
    .drop_duplicates(["competition_id", "season", "fotmob_player_id"]),
    tm_side,
    lineage_all := build_team_lineage(
        tm_clubs,
        {
            "fotmob": fm[["competition_id", "team_name"]].drop_duplicates(),
            "sofascore": ss[["competition_id", "team_name"]].drop_duplicates(),
            "understat": us[["competition_id", "team"]]
            .drop_duplicates()
            .rename(columns={"team": "team_name"}),
        },
        load_overrides("teams"),
    ),
    people,
).drop_duplicates("provider_id")[["provider_id", "tm_player_id"]]
k = kp_season.copy()
k["tm_player_id"] = (
    k.player_id.astype(int).astype(str).map(us_ids.set_index("provider_id").tm_player_id)
)
g = gp.copy()
g["tm_player_id"] = (
    g.fotmob_player_id.astype(int).astype(str).map(fm_ids.set_index("provider_id").tm_player_id)
)
g["gp_per90"] = g.goals_prevented / g.minutes_played * 90
sv = wr[wr.role == "GK"][["competition_id", "season", "tm_player_id", "saves", "goals_conceded"]]
three = k.merge(
    g[["competition_id", "season", "tm_player_id", "gp_per90"]],
    on=["competition_id", "season", "tm_player_id"],
).merge(sv, on=["competition_id", "season", "tm_player_id"])

print(len(three), "keeper-seasons with all three measures")
print("correlations between measures:")
print(three[["prevented_per90", "gp_per90", "saves"]].corr().round(2).to_string())
z = (
    three[["prevented_per90", "gp_per90"]] - three[["prevented_per90", "gp_per90"]].mean()
) / three[["prevented_per90", "gp_per90"]].std()
three["combined"] = z.mean(axis=1)
nxt = three.assign(season=three.season - 1)[
    ["competition_id", "season", "tm_player_id", "prevented_per90", "gp_per90", "combined"]
]
pairs = three.merge(nxt, on=["competition_id", "season", "tm_player_id"], suffixes=("", "_next"))

print(
    "year-to-year r on the same keepers:",
    {
        c: round(pairs[c].corr(pairs[f"{c}_next"]), 3)
        for c in ["prevented_per90", "gp_per90", "combined"]
    },
    "| n =",
    len(pairs),
)

486 keeper-seasons with all three measures
correlations between measures:
                 prevented_per90  gp_per90  saves
prevented_per90             1.00      0.45  -0.20
gp_per90                    0.45      1.00   0.01
saves                      -0.20      0.01   1.00
year-to-year r on the same keepers: {'prevented_per90': np.float64(0.42), 'gp_per90': np.float64(0.261), 'combined': np.float64(0.39)} | n = 245


In [20]:
# do defensive actions travel? same-role movers, per-90 actions at club A (season s) vs club B (s+1)

ss_clubs = ss[["competition_id", "season", "sofascore_player_id", "team_name"]].copy()
ss_clubs["club_id"] = ss_clubs.merge(
    lineage_all[lineage_all.provider == "sofascore"][["competition_id", "team_name", "club_id"]],
    on=["competition_id", "team_name"],
    how="left",
).club_id.values
wr_club = wr.merge(
    ss_clubs[["competition_id", "season", "sofascore_player_id", "club_id"]],
    on=["competition_id", "season", "sofascore_player_id"],
)
wr_club = wr_club[wr_club.minutes >= quantities.MIN_MINUTES].drop_duplicates(
    ["season", "tm_player_id", "role"]
)
ACTIONS = ["tackles", "interceptions", "clearances", "recoveries", "accurate_passes", "dribbles"]
nxt = wr_club.assign(season=wr_club.season - 1)[
    ["season", "tm_player_id", "role", "club_id"] + ACTIONS
].rename(columns={"club_id": "club_next", **{a: f"{a}_next" for a in ACTIONS}})
pairs = wr_club.merge(nxt, on=["season", "tm_player_id", "role"])
rows = {}

for role in ["CB", "FB", "CM"]:
    for label, mask in [
        ("stayers", pairs.club_id == pairs.club_next),
        ("movers", pairs.club_id != pairs.club_next),
    ]:
        sub = pairs[(pairs.role == role) & mask]
        rows[(role, label)] = {a: round(sub[a].corr(sub[f"{a}_next"]), 2) for a in ACTIONS} | {
            "n": len(sub)
        }

print(pd.DataFrame(rows).T.to_string())

            tackles  interceptions  clearances  recoveries  accurate_passes  dribbles       n
CB stayers     0.59           0.64        0.67        0.65             0.84      0.56  1892.0
   movers      0.47           0.55        0.55        0.48             0.49      0.54   541.0
FB stayers     0.63           0.65        0.64        0.60             0.87      0.72  1562.0
   movers      0.53           0.64        0.54        0.39             0.53      0.69   432.0
CM stayers     0.66           0.65        0.69        0.55             0.87      0.74  1987.0
   movers      0.59           0.61        0.56        0.49             0.62      0.72   604.0


### Step 6 — what we got

**Defensive value cannot be measured from results at this sample size — a negative result to
report, not to hide.** A proper on/off (17,327 player-club-seasons, at least 5 matches played and
5 missed) repeats year to year with r between −0.03 and +0.08 in every role, for players who stay
and for players who move; its season-level noise (sd 0.32 xG per 90) swamps any player effect.
Together with Step 2 this ends the spec's "plus-minus is required for defenders": no on-pitch
xG-difference construction carries a player's defensive contribution in this data.

**What does travel is what a defender does.** Tackles, interceptions, clearances and recoveries
per 90 keep r 0.47–0.64 after a move (0.59–0.69 for stayers) for CB, FB and CM — they are player
traits, not team artefacts. Decision: the contribution number is expected output for every role
(Step 2); defenders and midfielders additionally carry a **defensive activity profile** (the four
actions per 90, standardised within role), used for fit and similarity — and the writeup says
plainly that activity is measured while value is not. Ported: `contribution.DEFENSIVE_ACTIONS`.

**Keepers stay in, on the Understat proxy alone.** On-target xG faced minus goals conceded per
90 repeats with r 0.34 on 889 keeper-seasons (0.42 on the 245 that also have FotMob data), above
the 0.3 bar. FotMob's goals prevented repeats at 0.26–0.36 and correlates 0.45 with the proxy;
averaging the two (0.39) does not beat the proxy; Sofascore saves per 90 is a volume count
(r −0.20 with the proxy), not a quality measure. Rejected: FotMob as the primary, the average,
saves. The proxy is negative for everyone (mean −0.57 per 90) because on-target xG is measured
before the shot — only the *ranking* means something, and it gets the Step 9 shrinkage. Ported:
`scout.models.keepers.prevented_per90`.

### Step 6 check — the package reproduces the keeper proxy

**What?** `scout.models.keepers` on the same seasons; count, mean and year-to-year r must match.

In [21]:
from scout.models import keepers as keepers_model

shots_keyed = shots.assign(competition_id=shots.league.map(LEAGUE_TO_COMP))
packaged = keepers_model.prevented_per90(pm, shots_keyed, team_game)
packaged = packaged[packaged.minutes >= quantities.MIN_MINUTES]
nxt = packaged.assign(season=packaged.season - 1)[
    ["competition_id", "season", "player_id", "prevented_per90"]
].rename(columns={"prevented_per90": "next"})
pairs = packaged.merge(nxt, on=["competition_id", "season", "player_id"])

print(
    len(packaged),
    "keeper-seasons (above: 1,591) | mean",
    round(packaged.prevented_per90.mean(), 3),
    "(above: -0.565) | y2y r",
    round(pairs.prevented_per90.corr(pairs.next), 3),
    "(above: 0.336)",
)

1591 keeper-seasons (above: 1,591) | mean -0.565 (above: -0.565) | y2y r 0.336 (above: 0.336)


## Step 7 — What happens to a player's numbers when he changes league?

**What?** From movers — the same player with 600+ minutes in league A one season and league B the
next — the typical ratio of output after to before, per league pair, controlling for age. Understat
covers only the Big 5, so the cross-league measure is FotMob's xG + xA per 90 on both sides (one
provider, one definition; Phase 1 showed it equals Sofascore's), with Understat as the cross-check
for Big-5 ↔ Big-5 moves. Pairs with fewer than 30 movers pool by tier (Big 5 vs feeder).

**Why?** The candidate pool is the feeder leagues; a striker's 0.6 per 90 in the Netherlands is
not 0.6 per 90 in England. The spec's test: do held-out movers land inside the stated intervals?

In [22]:
players_tm = tm_loader.load_table("players")[["player_id", "date_of_birth"]].rename(
    columns={"player_id": "tm_player_id"}
)
players_tm["tm_player_id"] = players_tm.tm_player_id.astype(str)
fm_xg = (
    fm[fm.stat.isin(["expected_goals", "expected_assists", "mins_played"])]
    .pivot_table(
        index=["competition_id", "season", "fotmob_player_id"],
        columns="stat",
        values="stat_value",
        aggfunc="first",
    )
    .reset_index()
)
fm_xg = fm_xg.rename(columns={"mins_played": "minutes"}).dropna(subset=["minutes"])
fm_xg = fm_xg[fm_xg.minutes >= quantities.MIN_MINUTES].fillna(
    {"expected_goals": 0.0, "expected_assists": 0.0}
)
fm_xg["output"] = (fm_xg.expected_goals + fm_xg.expected_assists) / fm_xg.minutes * 90
fm_xg["tm_player_id"] = (
    fm_xg.fotmob_player_id.astype(int).astype(str).map(fm_ids.set_index("provider_id").tm_player_id)
)
fm_xg = (
    fm_xg.dropna(subset=["tm_player_id"])
    .sort_values("minutes", ascending=False)
    .drop_duplicates(["season", "tm_player_id"])
)
fm_xg = fm_xg.merge(players_tm, on="tm_player_id", how="left")
fm_xg["age"] = (
    fm_xg.season + 1 - pd.to_datetime(fm_xg.date_of_birth).dt.year
)  # age in the season's spring

print(
    len(fm_xg),
    "FotMob player-seasons ≥600 min with a Transfermarkt id |",
    fm_xg.groupby("competition_id").season.nunique().to_dict(),
)
nxt = fm_xg.assign(season=fm_xg.season - 1)[
    ["season", "tm_player_id", "competition_id", "output"]
].rename(columns={"competition_id": "league_to", "output": "output_after"})
moves = fm_xg.merge(nxt, on=["season", "tm_player_id"])
moves = moves[moves.competition_id != moves.league_to].copy()
moves["log_ratio"] = np.log(
    (moves.output_after + 0.02) / (moves.output + 0.02)
)  # +0.02 keeps zero-output seasons finite

print(
    len(moves),
    "cross-league movers |",
    moves.groupby(["competition_id", "league_to"])
    .size()
    .sort_values(ascending=False)
    .head(12)
    .to_dict(),
)

32518 FotMob player-seasons ≥600 min with a Transfermarkt id | {'A1': 8, 'BE1': 8, 'BRA1': 10, 'C1': 8, 'DK1': 9, 'ES1': 10, 'FR1': 10, 'GB1': 10, 'IT1': 10, 'NL1': 9, 'PO1': 9, 'TR1': 8}
2374 cross-league movers | {('FR1', 'GB1'): 97, ('ES1', 'GB1'): 86, ('FR1', 'IT1'): 86, ('IT1', 'GB1'): 72, ('GB1', 'ES1'): 72, ('GB1', 'IT1'): 71, ('FR1', 'ES1'): 61, ('ES1', 'IT1'): 58, ('PO1', 'TR1'): 57, ('FR1', 'TR1'): 54, ('IT1', 'FR1'): 54, ('IT1', 'TR1'): 50}


In [23]:
BIG5 = set(config.BIG5)
moves["tier_from"] = np.where(moves.competition_id.isin(BIG5), "big5", "feeder")
moves["tier_to"] = np.where(moves.league_to.isin(BIG5), "big5", "feeder")

# age control: residualise the log ratio on age bands across all movers
moves["age_band"] = pd.cut(
    moves.age, [15, 21, 24, 27, 30, 45], labels=["≤21", "22-24", "25-27", "28-30", "31+"]
)
age_effect = moves.groupby("age_band", observed=True).log_ratio.mean()

print("mean log ratio by age band (all movers):", age_effect.round(3).to_dict())
moves["adj"] = (
    moves.log_ratio - moves.age_band.map(age_effect).astype(float) + moves.log_ratio.mean()
)


def factor(frame, col="adj", n_boot=500, seed=0):
    rng = np.random.default_rng(seed)
    x = frame[col].to_numpy()
    med = np.median(x)
    boots = [np.median(rng.choice(x, len(x))) for _ in range(n_boot)]
    return (
        round(float(np.exp(med)), 3),
        round(float(np.exp(np.percentile(boots, 10))), 3),
        round(float(np.exp(np.percentile(boots, 90))), 3),
    )


rows = []

for (a, b), g in moves.groupby(["competition_id", "league_to"]):
    if len(g) >= 30:
        f, lo, hi = factor(g)
        rows.append((a, b, len(g), f, lo, hi))

pairs_table = pd.DataFrame(rows, columns=["from", "to", "n", "factor", "p10", "p90"]).sort_values(
    "n", ascending=False
)

print(
    "league pairs with ≥30 movers (factor = multiplier on xG+xA per 90 after the move, age-adjusted, 80% bootstrap interval):"
)

print(pairs_table.to_string(index=False))
print("\npooled by tier:")

for (a, b), g in moves.groupby(["tier_from", "tier_to"]):
    print(f"  {a} → {b}: n={len(g)}, factor {factor(g)}")

mean log ratio by age band (all movers): {'≤21': 0.263, '22-24': 0.135, '25-27': 0.176, '28-30': 0.053, '31+': 0.197}


league pairs with ≥30 movers (factor = multiplier on xG+xA per 90 after the move, age-adjusted, 80% bootstrap interval):
from  to  n  factor   p10   p90
 FR1 GB1 97   0.961 0.899 0.979
 FR1 IT1 86   0.979 0.914 1.011
 ES1 GB1 86   1.020 0.979 1.032
 GB1 ES1 72   1.057 1.020 1.095
 IT1 GB1 72   0.979 0.929 0.979
 GB1 IT1 71   1.020 0.979 1.107
 FR1 ES1 61   0.979 0.979 0.979
 ES1 IT1 58   0.979 0.979 1.020
 PO1 TR1 57   1.020 0.987 1.097
 FR1 TR1 54   0.979 0.968 1.020
 IT1 FR1 54   1.020 0.979 1.020
 IT1 TR1 50   1.044 0.979 1.153
BRA1 PO1 49   1.107 1.020 1.221
 IT1 ES1 48   0.999 0.968 1.096
 ES1 FR1 47   1.049 0.979 1.107
 GB1 FR1 47   1.107 1.107 1.197
 BE1 FR1 47   0.858 0.782 0.920
 GB1 TR1 39   1.076 0.979 1.137
 BE1 TR1 38   0.979 0.958 0.984
 NL1 BE1 35   1.020 0.979 1.163
 NL1 GB1 34   0.563 0.459 0.706
 PO1 FR1 33   1.020 0.979 1.030
 FR1 BE1 32   1.020 0.979 1.112
 BE1 IT1 32   0.648 0.602 0.827
 NL1 IT1 30   0.897 0.686 0.957

pooled by tier:
  big5 → big5: n=799, factor (

In [24]:
# Kill check: do held-out movers land inside the interval? Leave-one-season-out on the tier pools.

coverage = []

for (a, b), g in moves.groupby(["tier_from", "tier_to"]):
    for s in sorted(g.season.unique()):
        train, test = g[g.season != s], g[g.season == s]
        if len(train) < 30 or len(test) < 5:
            continue
        f, lo, hi = factor(train)
        # prediction interval for an individual mover: the factor ± the spread of individual ratios (10th-90th pct of adj residuals)
        spread = np.percentile(train.adj - np.median(train.adj), [10, 90])
        inside = ((test.adj >= np.log(f) + spread[0]) & (test.adj <= np.log(f) + spread[1])).mean()
        coverage.append((a, b, s, len(test), round(inside, 2)))

cov = pd.DataFrame(coverage, columns=["from", "to", "season", "n_test", "inside_80pct"])

print("held-out season coverage of the 80% individual interval, by tier pair:")
print(
    cov.groupby(["from", "to"])
    .apply(
        lambda d: pd.Series(
            {
                "seasons": len(d),
                "movers": d.n_test.sum(),
                "coverage": round(np.average(d.inside_80pct, weights=d.n_test), 3),
            }
        )
    )
    .to_string()
)

# Big-5 ↔ Big-5 cross-check with Understat npxG + xA (different provider, penalty-free)
us_out = per90[per90.minutes >= quantities.MIN_MINUTES].copy()
us_out["output"] = us_out.npxg + us_out.xa
us_out = us_out.sort_values("minutes", ascending=False).drop_duplicates(["season", "player_id"])
nxt = us_out.assign(season=us_out.season - 1)[
    ["season", "player_id", "competition_id", "output"]
].rename(columns={"competition_id": "league_to", "output": "output_after"})
us_moves = us_out.merge(nxt, on=["season", "player_id"])
us_moves = us_moves[us_moves.competition_id != us_moves.league_to]
us_moves["log_ratio"] = np.log((us_moves.output_after + 0.02) / (us_moves.output + 0.02))

print(
    f"\nUnderstat Big-5 ↔ Big-5 movers: n={len(us_moves)}, factor {factor(us_moves, 'log_ratio')} | FotMob on the same tier pair: {factor(moves[(moves.tier_from == 'big5') & (moves.tier_to == 'big5')], 'log_ratio')}"
)

held-out season coverage of the 80% individual interval, by tier pair:
               seasons  movers  coverage
from   to                               
big5   big5        9.0   799.0     0.796
       feeder      9.0   399.0     0.794
feeder big5        8.0   521.0     0.743
       feeder      8.0   654.0     0.772

Understat Big-5 ↔ Big-5 movers: n=1259, factor (0.988, 0.968, 1.0) | FotMob on the same tier pair: (1.0, 1.0, 1.0)


**Something is wrong with the table above.** The medians sit on the same two values (0.979 and
1.020) and one pair has an interval of zero width — many movers must share an identical ratio.
The likely cause is FotMob's *total* stat list being absent for some players and filled as 0
(the open "absence semantics" question from Phase 1). Next cells: check how often the list is
absent, then redo the factors on FotMob's per-90 xG + xA list (which is missing when absent, never
0), only for movers with real output before the move, and compare the ratio form with a
difference form.

In [25]:
fm_raw = (
    fm[
        fm.stat.isin(
            ["expected_goals", "_expected_goals_and_expected_assists_per_90", "mins_played"]
        )
    ]
    .pivot_table(
        index=["competition_id", "season", "fotmob_player_id"],
        columns="stat",
        values="stat_value",
        aggfunc="first",
    )
    .reset_index()
)
fm_raw = fm_raw[fm_raw.mins_played >= quantities.MIN_MINUTES]

print(
    "FotMob player-seasons ≥600 min:",
    len(fm_raw),
    "| expected_goals present:",
    f"{fm_raw.expected_goals.notna().mean():.1%}",
    "| xG+xA per-90 list present:",
    f"{fm_raw._expected_goals_and_expected_assists_per_90.notna().mean():.1%}",
)

print(
    "expected_goals present by league:",
    fm_raw.groupby("competition_id")
    .expected_goals.apply(lambda c: round(c.notna().mean(), 2))
    .to_dict(),
)

print(
    "share of movers above with output exactly equal before and after:",
    f"{(moves.output == moves.output_after).mean():.1%}",
    "| with output 0 before:",
    f"{(moves.output == 0).mean():.1%}",
)

FotMob player-seasons ≥600 min: 35768 | expected_goals present: 60.3% | xG+xA per-90 list present: 54.1%
expected_goals present by league: {'A1': 0.71, 'BE1': 0.73, 'BRA1': 0.52, 'C1': 0.74, 'DK1': 0.65, 'ES1': 0.57, 'FR1': 0.57, 'GB1': 0.58, 'IT1': 0.57, 'NL1': 0.64, 'PO1': 0.62, 'TR1': 0.52}
share of movers above with output exactly equal before and after: 26.4% | with output 0 before: 38.0%


In [26]:
# Redo on the per-90 list; movers with ≥ 0.10 xG+xA per 90 before the move (a real attacking output to convert)

fm90 = fm_raw.rename(
    columns={"_expected_goals_and_expected_assists_per_90": "output", "mins_played": "minutes"}
).dropna(subset=["output"])
fm90["tm_player_id"] = (
    fm90.fotmob_player_id.astype(int).astype(str).map(fm_ids.set_index("provider_id").tm_player_id)
)
fm90 = (
    fm90.dropna(subset=["tm_player_id"])
    .sort_values("minutes", ascending=False)
    .drop_duplicates(["season", "tm_player_id"])
    .merge(players_tm, on="tm_player_id", how="left")
)
fm90["age"] = fm90.season + 1 - pd.to_datetime(fm90.date_of_birth).dt.year
nxt = fm90.assign(season=fm90.season - 1)[
    ["season", "tm_player_id", "competition_id", "output"]
].rename(columns={"competition_id": "league_to", "output": "output_after"})
mv = fm90.merge(nxt, on=["season", "tm_player_id"])
mv = mv[(mv.competition_id != mv.league_to) & (mv.output >= 0.10)].copy()
mv["log_ratio"] = np.log(mv.output_after.clip(lower=0.02) / mv.output)
mv["diff"] = mv.output_after - mv.output
mv["tier_from"] = np.where(mv.competition_id.isin(BIG5), "big5", "feeder")
mv["tier_to"] = np.where(mv.league_to.isin(BIG5), "big5", "feeder")
mv["age_band"] = pd.cut(
    mv.age, [15, 21, 24, 27, 30, 45], labels=["≤21", "22-24", "25-27", "28-30", "31+"]
)

for col in ["log_ratio", "diff"]:
    eff = mv.groupby("age_band", observed=True)[col].mean()
    mv[f"{col}_adj"] = mv[col] - mv.age_band.map(eff).astype(float) + mv[col].mean()

print(
    len(mv),
    "movers with ≥0.10 per 90 before | age effect on log ratio:",
    mv.groupby("age_band", observed=True).log_ratio.mean().round(3).to_dict(),
)

print("\nleague pairs with ≥30 movers — ratio form (age-adjusted, 80% bootstrap):")
rows = [
    (a, b, len(g), *factor(g, "log_ratio_adj"))
    for (a, b), g in mv.groupby(["competition_id", "league_to"])
    if len(g) >= 30
]

print(
    pd.DataFrame(rows, columns=["from", "to", "n", "factor", "p10", "p90"])
    .sort_values("n", ascending=False)
    .to_string(index=False)
)

print("\npooled by tier — ratio form | difference form (per 90):")

for (a, b), g in mv.groupby(["tier_from", "tier_to"]):
    d = g.diff_adj.to_numpy()
    rng = np.random.default_rng(0)
    boots = [np.median(rng.choice(d, len(d))) for _ in range(500)]
    print(
        f"  {a} → {b}: n={len(g)} | ratio {factor(g, 'log_ratio_adj')} | diff {round(float(np.median(d)), 3)} [{round(float(np.percentile(boots, 10)), 3)}, {round(float(np.percentile(boots, 90)), 3)}]"
    )

718 movers with ≥0.10 per 90 before | age effect on log ratio: {'≤21': -0.066, '22-24': -0.246, '25-27': -0.18, '28-30': -0.25, '31+': -0.224}

league pairs with ≥30 movers — ratio form (age-adjusted, 80% bootstrap):
from  to  n  factor   p10   p90
 FR1 GB1 35   0.672 0.606 0.734

pooled by tier — ratio form | difference form (per 90):
  big5 → big5: n=224 | ratio (0.849, 0.809, 0.885) | diff -0.035 [-0.045, -0.023]
  big5 → feeder: n=102 | ratio (1.211, 1.178, 1.305) | diff 0.069 [0.039, 0.083]
  feeder → big5: n=195 | ratio (0.669, 0.631, 0.711) | diff -0.093 [-0.121, -0.071]
  feeder → feeder: n=197 | ratio (0.902, 0.872, 0.941) | diff -0.023 [-0.031, -0.011]


In [27]:
# Held-out coverage and error, ratio vs difference, by tier pair (leave-one-season-out)


def holdout(frame, form):
    col = "log_ratio_adj" if form == "ratio" else "diff_adj"
    out = []
    for s in sorted(frame.season.unique()):
        train, test = frame[frame.season != s], frame[frame.season == s]
        if len(train) < 30 or len(test) < 5:
            continue
        centre = np.median(train[col])
        spread = np.percentile(train[col] - centre, [10, 90])
        if form == "ratio":
            pred = test.output * np.exp(centre)
            lo = test.output * np.exp(centre + spread[0])
            hi = test.output * np.exp(centre + spread[1])
        else:
            pred = test.output + centre
            lo = test.output + centre + spread[0]
            hi = test.output + centre + spread[1]
        out.append(
            (
                len(test),
                ((test.output_after >= lo) & (test.output_after <= hi)).mean(),
                np.abs(test.output_after - pred).mean(),
            )
        )
    n = sum(o[0] for o in out)
    return {
        "movers": n,
        "coverage_80": round(sum(o[0] * o[1] for o in out) / n, 3),
        "mae": round(sum(o[0] * o[2] for o in out) / n, 4),
    }


print("held-out, by tier pair:")

for (a, b), g in mv.groupby(["tier_from", "tier_to"]):
    print(
        f"  {a} → {b}: ratio {holdout(g, 'ratio')} | diff {holdout(g, 'diff')} | no-factor baseline mae {round(float(np.abs(g.output_after - g.output).mean()), 4)}"
    )

held-out, by tier pair:
  big5 → big5: ratio {'movers': 223, 'coverage_80': np.float64(0.794), 'mae': np.float64(0.0975)} | diff {'movers': 223, 'coverage_80': np.float64(0.839), 'mae': np.float64(0.1028)} | no-factor baseline mae 0.11
  big5 → feeder: ratio {'movers': 102, 'coverage_80': np.float64(0.804), 'mae': np.float64(0.1217)} | diff {'movers': 102, 'coverage_80': np.float64(0.794), 'mae': np.float64(0.1267)} | no-factor baseline mae 0.1408
  feeder → big5: ratio {'movers': 195, 'coverage_80': np.float64(0.774), 'mae': np.float64(0.0876)} | diff {'movers': 195, 'coverage_80': np.float64(0.795), 'mae': np.float64(0.1213)} | no-factor baseline mae 0.1488
  feeder → feeder: ratio {'movers': 197, 'coverage_80': np.float64(0.782), 'mae': np.float64(0.1052)} | diff {'movers': 197, 'coverage_80': np.float64(0.787), 'mae': np.float64(0.1104)} | no-factor baseline mae 0.111


### Step 7 — what we got

**Two findings first.** (1) FotMob's *total* lists (xG, xA, …) are absent for 40% of
player-seasons with 600+ minutes, and absence is not zero: 38% of the first mover set had
"output 0 before" and 26% an identical number on both sides, which produced the spikes. Decision
on the open Phase 1 question: a missing FotMob value is NaN, never 0, for every stat. (2) The
per-90 xG + xA list (present for 54%, missing when absent) with a floor of 0.10 per 90 before the
move gives 718 clean movers across all leagues.

**Chosen: the ratio form, pooled by tier, plus any league pair with 30+ movers.** Factors are
age-adjusted median multipliers on xG + xA per 90 (80% interval on the median; the interval for one
mover is the 10th–90th percentile of movers' ratios):

| pair | movers | factor | held-out 80% coverage | held-out error (no factor) |
|---|---|---|---|---|
| feeder → Big 5 | 195 | **0.67** [0.63, 0.71] | 0.77 | 0.088 (0.149) |
| Big 5 → feeder | 102 | 1.21 [1.18, 1.31] | 0.80 | 0.122 (0.141) |
| Big 5 → Big 5 | 224 | 0.85 [0.81, 0.89] | 0.79 | 0.098 (0.110) |
| feeder → feeder | 197 | 0.90 [0.87, 0.94] | 0.78 | 0.105 (0.111) |
| FR1 → GB1 (only pair with 30+) | 35 | 0.67 [0.61, 0.73] | – | – |

- The Big-5 → Big-5 factor of 0.85 is not a league effect: it is regression to the mean — players
  move after a good season. Every pair carries it, so the league-specific part of feeder → Big 5
  is about 0.67 / 0.85 ≈ 0.79; the model applies the raw factor because a candidate scouted on a
  good feeder season is exactly a past mover.
- Ratio beats difference on held-out error in every tier (feeder → Big 5: 0.088 vs 0.121), and
  both beat "no factor". Coverage 0.77–0.80 against the nominal 0.80 passes the test. Rejected:
  the difference form; per-league-pair factors below 30 movers.
- The age adjustment is negative for every band above 21 (−0.18 to −0.25 in log): older movers
  lose more; Phase 3's trajectory model owns that.

Ported: `scout.models.league_factors`.

### Step 7 check — the package reproduces the tier table

**What?** `scout.models.league_factors.tier_factors` on the same movers; factors and counts must
match.

In [28]:
from scout.models import league_factors

table = league_factors.tier_factors(
    mv[["competition_id", "league_to", "output", "output_after", "age"]], set(config.BIG5)
)

print(table.round(3).to_string(index=False))
print(
    "(above: feeder→big5 0.669 [0.631, 0.711] n=195; big5→feeder 1.211; big5→big5 0.849; feeder→feeder 0.902; FR1→GB1 0.672 n=35)"
)

  from     to  movers  factor   p10   p90  spread_lo  spread_hi
  big5   big5     224   0.849 0.809 0.885     -0.589      0.614
  big5 feeder     102   1.211 1.178 1.305     -0.858      0.546
feeder   big5     195   0.669 0.631 0.711     -0.572      0.470
feeder feeder     197   0.902 0.872 0.941     -0.708      0.543
   FR1    GB1      35   0.672 0.606 0.734     -0.560      0.441
(above: feeder→big5 0.669 [0.631, 0.711] n=195; big5→feeder 1.211; big5→big5 0.849; feeder→feeder 0.902; FR1→GB1 0.672 n=35)


## Step 8 — What does a freely available player contribute?

**What?** Two definitions per role and season: (i) the median output of cheap signings in their
first season (free transfers, or a fee under that season's 25th percentile of paid fees); (ii) a
low percentile (the 20th) of every regular in the role. Test (from the spec): the one that is
more stable from season to season.

**Why?** "Surplus" = contribution above what a club could get for nothing. A replacement level
that jumps around from year to year would make surplus meaningless.

In [29]:
# signings: a club-season-role row whose player moved to that club in that transfer window

us_tm = us_ids.set_index("provider_id").tm_player_id
club_rows = stint.copy()
club_rows["tm_player_id"] = club_rows.player_id.astype(int).astype(str).map(us_tm)
club_rows["club_id"] = club_rows.merge(
    lineage_all[lineage_all.provider == "understat"][["competition_id", "team_name", "club_id"]]
    .rename(columns={"team_name": "team"})
    .merge(
        us[["competition_id", "team", "team_id"]].drop_duplicates(), on=["competition_id", "team"]
    )[["competition_id", "team_id", "club_id"]],
    on=["competition_id", "team_id"],
    how="left",
).club_id.values
market_moves = market.build()  # `moves` was reused for the movers table in Step 7
mv_in = market_moves[market_moves.kind.isin(["paid", "free"])].copy()
mv_in["transfer_season"] = ("20" + mv_in.transfer_season.str[:2]).astype(int)
mv_in["season"] = mv_in.transfer_season - pd.to_datetime(mv_in.transfer_date).dt.month.isin(
    [1, 2, 3]
).astype(int)  # winter signings count for the season already running
p25 = mv_in[mv_in.kind == "paid"].groupby("transfer_season").transfer_fee.quantile(0.25)
mv_in["cheap"] = (mv_in.kind == "free") | (mv_in.transfer_fee < mv_in.transfer_season.map(p25))
mv_in["tm_player_id"] = mv_in.player_id.astype(str)
signings = club_rows.merge(
    mv_in[["tm_player_id", "to_club_id", "season", "kind", "transfer_fee", "cheap"]].rename(
        columns={"to_club_id": "club_id"}
    ),
    on=["tm_player_id", "club_id", "season"],
)

print(
    len(signings),
    "signings with ≥600 min in their first season |",
    signings.kind.value_counts().to_dict(),
    "| cheap:",
    int(signings.cheap.sum()),
)

print("paid-fee 25th percentile by season (M€):", p25.div(1e6).round(2).loc[2015:2025].to_dict())

2961 signings with ≥600 min in their first season | {'paid': 2359, 'free': 602} | cheap: 635
paid-fee 25th percentile by season (M€): {2015: 0.49, 2016: 0.5, 2017: 0.5, 2018: 0.45, 2019: 0.4, 2020: 0.3, 2021: 0.32, 2022: 0.45, 2023: 0.41, 2024: 0.4, 2025: 0.48}


In [30]:
levels = []

for role, g in stint.groupby("role"):
    for season, gs in g.groupby("season"):
        cheap = signings[(signings.role == role) & (signings.season == season) & signings.cheap]
        levels.append(
            {
                "role": role,
                "season": season,
                "cheap_median": cheap.raw.median(),
                "n_cheap": len(cheap),
                "p20_all": gs.raw.quantile(0.20),
                "p25_all": gs.raw.quantile(0.25),
                "n_all": len(gs),
            }
        )

levels = pd.DataFrame(levels)
summary = (
    levels[levels.season.between(2015, 2024)]
    .groupby("role")
    .agg(
        cheap_mean=("cheap_median", "mean"),
        cheap_cv=("cheap_median", lambda x: x.std() / x.mean()),
        n_cheap_per_season=("n_cheap", "mean"),
        p20_mean=("p20_all", "mean"),
        p20_cv=("p20_all", lambda x: x.std() / x.mean()),
        p25_mean=("p25_all", "mean"),
        p25_cv=("p25_all", lambda x: x.std() / x.mean()),
        n_all=("n_all", "mean"),
    )
    .round(3)
)

print(
    "replacement level per role, 2015-16 → 2024-25: mean level and year-to-year coefficient of variation"
)

print(summary.to_string())
print(
    "\ncorrelation across seasons between the cheap-signing median and the 20th percentile, by role:"
)

print(
    levels[levels.season.between(2015, 2024)]
    .groupby("role")
    .apply(lambda d: round(d.cheap_median.corr(d.p20_all), 2))
    .to_dict()
)

print(
    "\nwhere does the cheap-signing median sit in the role's distribution? (percentile of all regulars, by role)"
)

print(
    {
        role: round(
            float((stint[stint.role == role].raw < summary.loc[role, "cheap_mean"]).mean()), 2
        )
        for role in summary.index
    }
)

replacement level per role, 2015-16 → 2024-25: mean level and year-to-year coefficient of variation
      cheap_mean  cheap_cv  n_cheap_per_season  p20_mean  p20_cv  p25_mean  p25_cv  n_all
role                                                                                     
CB         0.056     0.370                10.5     0.030   0.117     0.036   0.107  381.8
CM         0.178     0.453                10.3     0.074   0.125     0.085   0.123  420.8
FB         0.114     0.297                11.5     0.067   0.208     0.076   0.186  340.8
GK         0.001     1.337                 5.2     0.000     NaN     0.000     NaN  133.3
ST         0.448     0.267                 6.2     0.368   0.048     0.393   0.048  246.7
W          0.295     0.188                 9.1     0.246   0.089     0.264   0.092  398.7

correlation across seasons between the cheap-signing median and the 20th percentile, by role:
{'CB': 0.32, 'CM': 0.11, 'FB': 0.7, 'GK': nan, 'ST': 0.72, 'W': 0.56}

where does the

{'CB': 0.43, 'CM': 0.61, 'FB': 0.46, 'GK': 0.66, 'ST': 0.38, 'W': 0.33}


/var/folders/s1/ydqc1kqd1_l_3stqdlcsdygm0000gn/T/ipykernel_79873/444144643.py:27: RuntimeWarning: invalid value encountered in scalar divide
  p20_cv=("p20_all", lambda x: x.std() / x.mean()),
/var/folders/s1/ydqc1kqd1_l_3stqdlcsdygm0000gn/T/ipykernel_79873/444144643.py:29: RuntimeWarning: invalid value encountered in scalar divide
  p25_cv=("p25_all", lambda x: x.std() / x.mean()),
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


### Step 8 — what we got

**Chosen: the 20th percentile of regulars (600+ minutes) in the role, per season.** It varies by 5%
(ST) to 21% (FB) across seasons on 250–420 players per role-season. The cheap-signing median
(free, or a fee under €0.3–0.5m) varies by 19% to 134% on 5–12 signings a season — and it does not
even sit low: cheap signings who reach 600 minutes are at the 33rd (W) to 61st (CM) percentile of
their role, because only the ones who play are observed. A survivorship measure, not the freely
available level. Rejected: the cheap-signing median; the 25th percentile (same stability, less
conservative). Keepers: expected output is ~0 for every keeper, so their level is the 20th
percentile of the Step 6 proxy. Surplus = contribution − the role-season level. Ported:
`scout.models.replacement`.

### Step 8 check — the package reproduces the p20 column

**What?** `scout.models.replacement.replacement_level` on the same seasons; means and variation
per role must match.

In [31]:
from scout.models import replacement

lv = replacement.replacement_level(stint[stint.season.between(2015, 2024)], "raw")
chk = lv.groupby("role").replacement_level.agg(["mean", lambda x: x.std() / x.mean()]).round(3)
chk.columns = ["p20_mean", "p20_cv"]

print(chk.to_string())
print(
    "(above: ST 0.368 / 0.048, W 0.246 / 0.089, CB 0.030 / 0.117, CM 0.074 / 0.125, FB 0.067 / 0.208)"
)

      p20_mean  p20_cv
role                  
CB        0.03   0.117
CM       0.074   0.125
FB       0.067   0.208
GK         0.0    <NA>
ST       0.368   0.048
W        0.246   0.089
(above: ST 0.368 / 0.048, W 0.246 / 0.089, CB 0.030 / 0.117, CM 0.074 / 0.125, FB 0.067 / 0.208)


/var/folders/s1/ydqc1kqd1_l_3stqdlcsdygm0000gn/T/ipykernel_79873/2218467958.py:4: RuntimeWarning: invalid value encountered in scalar divide
  chk = lv.groupby("role").replacement_level.agg(["mean", lambda x: x.std() / x.mean()]).round(3)


## Step 9 — How sure are we about a player's number?

**What?** Two candidates for the interval around a season's contribution: (a) resample the
player's matches within the season ("bootstrap"), no shrinkage; (b) shrink the season toward the
role average in proportion to how noisy it is (few matches → more shrinkage), and widen the
interval to include next season's noise. Test: does next season's actual number fall inside the
stated 80% and 95% intervals, per role? And does the shrunk number predict next season better than
the raw one?

**Why?** The spec requires every shown interval to be calibrated. An interval that says 80% and
covers 55% is worse than no interval.

In [32]:
rng = np.random.default_rng(0)
match_rows = pm.dropna(subset=["role"])[
    ["competition_id", "season", "player_id", "role", "minutes", "npxg", "xa"]
].copy()  # `rows` was reused in Step 7
match_rows["out"] = match_rows.npxg + match_rows.xa
keys = ["competition_id", "season", "player_id", "role"]
grp = match_rows.groupby(keys)
season_pt = grp.agg(
    minutes=("minutes", "sum"), out=("out", "sum"), matches=("out", "size")
).reset_index()
season_pt = season_pt[season_pt.minutes >= quantities.MIN_MINUTES].copy()
season_pt["per90"] = season_pt.out / season_pt.minutes * 90


def boot_sd(g, n_boot=200):
    out, mins = g.out.to_numpy(), g.minutes.to_numpy()
    n = len(out)
    idx = rng.integers(0, n, size=(n_boot, n))
    return float(np.std(out[idx].sum(axis=1) / mins[idx].sum(axis=1) * 90))


sds = grp.apply(boot_sd).rename("boot_sd").reset_index()
season_pt = season_pt.merge(sds, on=keys)

print(
    len(season_pt),
    "player-season-roles ≥600 min | median bootstrap sd by role:",
    season_pt.groupby("role").boot_sd.median().round(3).to_dict(),
)
nxt = season_pt.assign(season=season_pt.season - 1)[
    ["season", "player_id", "role", "per90"]
].rename(columns={"per90": "next"})
pairs9 = season_pt.merge(nxt, on=["season", "player_id", "role"])

print(len(pairs9), "pairs with a next season in the same role")

23070 player-season-roles ≥600 min | median bootstrap sd by role: {'CB': 0.027, 'CM': 0.043, 'FB': 0.041, 'GK': 0.0, 'ST': 0.106, 'W': 0.084}
12989 pairs with a next season in the same role


In [33]:
# empirical Bayes per role: tau^2 = Cov(this, next) (true-talent variance), noise = bootstrap variance

results = {}
shrunk_cols = {}

for role, g in pairs9.groupby("role"):
    mu = season_pt[season_pt.role == role].per90.mean()
    tau2 = max(float(np.cov(g.per90, g.next)[0, 1]), 1e-6)
    noise = g.boot_sd**2
    k = tau2 / (tau2 + noise)
    post_mean = mu + k * (g.per90 - mu)
    post_var = tau2 * noise / (tau2 + noise)
    pred_sd = np.sqrt(post_var + noise)  # next season is observed with its own noise
    z80, z95 = 1.2816, 1.96
    cov = lambda centre, sd, z: float(
        ((g.next >= centre - z * sd) & (g.next <= centre + z * sd)).mean()
    )
    results[role] = {
        "n": len(g),
        "tau": round(np.sqrt(tau2), 3),
        "median k": round(float(k.median()), 2),
        "raw r": round(g.per90.corr(g.next), 3),
        "shrunk r": round(post_mean.corr(g.next), 3),
        "raw mae": round(float((g.per90 - g.next).abs().mean()), 4),
        "shrunk mae": round(float((post_mean - g.next).abs().mean()), 4),
        "boot 80%": round(cov(g.per90, g.boot_sd, z80), 3),
        "boot 95%": round(cov(g.per90, g.boot_sd, z95), 3),
        "EB 80%": round(cov(post_mean, pred_sd, z80), 3),
        "EB 95%": round(cov(post_mean, pred_sd, z95), 3),
    }
    shrunk_cols[role] = (mu, tau2)

print(pd.DataFrame(results).T.to_string())
print(
    "\nnominal: 80% / 95%. Bootstrap-only intervals ignore the true year-to-year change; EB adds next season's noise."
)

         n    tau  median k  raw r  shrunk r  raw mae  shrunk mae  boot 80%  boot 95%  EB 80%  EB 95%
CB  2752.0  0.027      0.50  0.311     0.280   0.0425      0.0362     0.528     0.718   0.671   0.813
CM  2957.0  0.101      0.85  0.677     0.659   0.0711      0.0673     0.515     0.697   0.653   0.814
FB  2325.0  0.072      0.76  0.589     0.574   0.0628      0.0587     0.539     0.717   0.670   0.826
GK   972.0  0.001      1.00  0.073     0.117   0.0034      0.0025     0.632     0.707   0.682   0.711
ST  1589.0  0.165      0.72  0.565     0.550   0.1547      0.1423     0.549     0.744   0.705   0.876
W   2394.0  0.150      0.76  0.623     0.605   0.1261      0.1176     0.541     0.726   0.693   0.850

nominal: 80% / 95%. Bootstrap-only intervals ignore the true year-to-year change; EB adds next season's noise.


**Neither interval covers.** Talent also moves between seasons, and both intervals only carry
within-season noise. Fix, from the spec: widen. Next cell fits one widening factor per role on
seasons up to 2019-20 (so that 80% of training outcomes fall inside) and then checks coverage on
2020-21 onward, which the fit never saw.

In [34]:
def eb_fit(g, mu, tau2):
    noise = g.boot_sd**2
    k = tau2 / (tau2 + noise)
    return mu + k * (g.per90 - mu), np.sqrt(tau2 * noise / (tau2 + noise) + noise).clip(lower=1e-4)


calib = {}

for role, g in pairs9.groupby("role"):
    mu, tau2 = shrunk_cols[role]
    train, test = g[g.season <= 2019], g[g.season >= 2020]
    pm_tr, sd_tr = eb_fit(train, mu, tau2)
    z_train = (
        ((train.next - pm_tr) / sd_tr).astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    )
    inflate = float(
        np.percentile(np.abs(z_train), 80) / 1.2816
    )  # makes 80% of training z fall within ±1.2816
    pm_te, sd_te = eb_fit(test, mu, tau2)
    z_test = (
        ((test.next - pm_te) / (sd_te * inflate))
        .astype(float)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )
    calib[role] = {
        "n train": len(train),
        "n test": len(test),
        "inflation": round(inflate, 2),
        "test 80%": round(float((np.abs(z_test) <= 1.2816).mean()), 3),
        "test 95%": round(float((np.abs(z_test) <= 1.96).mean()), 3),
        "median half-width 80% (per 90)": round(float((1.2816 * sd_te * inflate).median()), 3),
    }

print(pd.DataFrame(calib).T.to_string())
print("\nnominal 80% / 95% — out of sample, 2020-21 → 2024-25")

    n train  n test  inflation  test 80%  test 95%  median half-width 80% (per 90)
CB   1486.0  1266.0       1.44     0.791     0.885                           0.065
CM   1631.0  1326.0       1.47     0.801     0.906                           0.112
FB   1231.0  1094.0       1.40     0.804     0.896                           0.107
GK    531.0   441.0       4.76     0.751     0.776                           0.001
ST    909.0   680.0       1.22     0.796     0.921                           0.221
W    1368.0  1026.0       1.29     0.788     0.915                           0.197

nominal 80% / 95% — out of sample, 2020-21 → 2024-25


### Step 9 — what we got

**Chosen: shrink toward the role average, then widen the interval per role.** The point estimate
shrinks a season toward the role mean with a weight from the year-to-year covariance of the same
player (the "true talent" spread: 0.03 for CB up to 0.17 for ST) and the noise from resampling his
matches; the median weight kept is 0.50 (CB) to 0.85 (CM). The shrunk number beats the raw per 90
at predicting next season in every role (error −5% to −15%).

Neither raw interval covers: resampling alone reaches 52–55% at nominal 80% (it ignores that talent
moves between seasons), the shrunk one 65–71%. One widening factor per role, fitted on seasons up
to 2019-20 — 1.22 (ST), 1.29 (W), 1.40 (FB), 1.44 (CB), 1.47 (CM) — gives **out-of-sample coverage
on 2020-21 → 2024-25 of 79–80% at nominal 80% in every outfield role**, and 89–92% at nominal 95%:
the tails are heavier than a normal distribution, which is reported, not hidden (a fatter-tailed
interval is something to test in Phase 5, where the backtest quotes 95% bands). Typical 80%
half-widths: ±0.07 per 90 (CB) to ±0.22 (ST). Rejected: resampling alone; an unwidened interval;
a mixed model (no failure of the simpler method to justify its runtime and hardware
non-determinism).

The GK row here is not the keeper interval — expected output is ~0 for every keeper; the same
machinery on the Step 6 proxy is in the check below. Ported: `scout.models.intervals`.

### Step 9 check — the package reproduces the outfield table, and the keeper proxy gets the same treatment

**What?** `scout.models.intervals` on the same pairs (coverage per role must match), then the
shrink-and-widen machinery applied to the keeper proxy.

In [35]:
from scout.models import intervals

chk = {}

for role, g in pairs9.groupby("role"):
    if role == "GK":
        continue
    mu, tau2 = intervals.role_prior(g)
    train, test = g[g.season <= 2019], g[g.season >= 2020]
    fit_tr = intervals.shrink(train.per90, train.boot_sd, mu, tau2)
    infl = intervals.inflation(fit_tr.point, fit_tr.predictive_sd, train.next)
    fit_te = intervals.shrink(test.per90, test.boot_sd, mu, tau2)
    band80, band95 = (
        intervals.interval(fit_te.point, fit_te.predictive_sd, infl),
        intervals.interval(fit_te.point, fit_te.predictive_sd, infl, z=intervals.Z95),
    )
    chk[role] = {
        "inflation": round(infl, 2),
        "test 80%": round(float(((test.next >= band80.lo) & (test.next <= band80.hi)).mean()), 3),
        "test 95%": round(float(((test.next >= band95.lo) & (test.next <= band95.hi)).mean()), 3),
    }

print(pd.DataFrame(chk).T.to_string())
print("(above: CB 1.44 / 0.791 / 0.885 … ST 1.22 / 0.796 / 0.921)")

# keepers: the Step 6 proxy per match -> bootstrap -> shrink -> inflate, same criterion
kp_match = kp.assign(out=kp.xg_on_target - kp.goals)[
    ["competition_id", "season", "player_id", "minutes", "out"]
]
kp_sd = intervals.bootstrap_sd(
    kp_match, ["competition_id", "season", "player_id"], "out"
).reset_index()
kp9 = kp_season.merge(kp_sd, on=["competition_id", "season", "player_id"]).rename(
    columns={"prevented_per90": "per90"}
)
nxt = kp9.assign(season=kp9.season - 1)[["competition_id", "season", "player_id", "per90"]].rename(
    columns={"per90": "next"}
)
kp_pairs = kp9.merge(nxt, on=["competition_id", "season", "player_id"])
mu, tau2 = intervals.role_prior(kp_pairs)
train, test = kp_pairs[kp_pairs.season <= 2019], kp_pairs[kp_pairs.season >= 2020]
fit_tr = intervals.shrink(train.per90, train.boot_sd, mu, tau2)
infl = intervals.inflation(fit_tr.point, fit_tr.predictive_sd, train.next)
fit_te = intervals.shrink(test.per90, test.boot_sd, mu, tau2)
band = intervals.interval(fit_te.point, fit_te.predictive_sd, infl)

print(
    f"\nkeepers on the proxy: tau {np.sqrt(tau2):.3f}, median k {fit_te.k.median():.2f}, inflation {infl:.2f}, out-of-sample 80% coverage {((test.next >= band.lo) & (test.next <= band.hi)).mean():.3f} (n={len(test)}), shrunk MAE {np.abs(fit_te.point - test.next).mean():.4f} vs raw {np.abs(test.per90 - test.next).mean():.4f}"
)

    inflation  test 80%  test 95%
CB       1.44     0.791     0.885
CM       1.47     0.801     0.906
FB       1.41     0.806     0.896
ST       1.21     0.791     0.921
W        1.30     0.790     0.922
(above: CB 1.44 / 0.791 / 0.885 … ST 1.22 / 0.796 / 0.921)



keepers on the proxy: tau 0.125, median k 0.42, inflation 1.11, out-of-sample 80% coverage 0.818 (n=400), shrunk MAE 0.1598 vs raw 0.1929


## Step 10 — Can the wider profile earn weights? (the "Rodri question")

**What?** Within-role ranking by npxG + xA underrates players whose worth is build-up and
ball-winning (the owner's example: Rodri). Three tests, per role, all held out by season, to see
whether the full validated stat list earns data-fitted weights against outcomes we have:
(1) does the profile predict **next-season output** better than output history alone?
(2) does it predict **future playing time** (minutes are the best quality label managers give)?
(3) does it predict **future market value** beyond what the market model already uses?

**Why?** The earlier outcome designs for defensive value (plus-minus, with/without) failed, so
any weights would have been invented. Playing time and price are two outcomes those designs never
used. Whatever earns a weight gets into a per-role score; whatever does not stays profile-only —
either way the answer is a number, not an assurance.

In [36]:
from sklearn.ensemble import HistGradientBoostingRegressor

# one row per player-season: role, all validated per-90 quantities + work-rate, next-season targets
base = per90.sort_values("minutes", ascending=False).drop_duplicates(["player_id", "season"]).copy()
base["output"] = base.npxg + base.xa
UQ = ["npxg", "xa", "key_passes", "shots", "xg_chain", "xg_buildup"]
wr_cols = [
    c
    for c in [
        "tackles",
        "interceptions",
        "recoveries",
        "clearances",
        "dribbles",
        "big_chances_created",
        "accurate_passes",
        "accurate_long_balls",
        "fouls",
        "possession_won_att_third_sofascore",
    ]
    if c in wr.columns
]
wr_slim = wr.loc[:, ~wr.columns.duplicated()][
    ["competition_id", "season", "tm_player_id"] + wr_cols
].drop_duplicates(["competition_id", "season", "tm_player_id"])
us_map = (
    us_ids.set_index("provider_id").tm_player_id if isinstance(us_ids, pd.DataFrame) else us_ids
)
base["tm_player_id"] = base.player_id.astype(int).astype(str).map(us_map)
XP = base.merge(wr_slim, on=["competition_id", "season", "tm_player_id"], how="left")
nxt = XP.assign(season=XP.season - 1)[["player_id", "season", "output", "minutes"]].rename(
    columns={"output": "output_next", "minutes": "minutes_next"}
)
XP = XP.merge(nxt, on=["player_id", "season"], how="left")
XP = XP[XP.minutes >= quantities.MIN_MINUTES]
print(
    len(XP),
    "player-seasons | with work-rate:",
    f"{XP[wr_cols[0]].notna().mean():.1%}",
    "| with a next season:",
    f"{XP.output_next.notna().mean():.1%}",
)
FEATS = UQ + wr_cols


def lfo_r(frame, features, target):
    rows = []
    for s in range(2018, 2025):
        tr = frame[(frame.season < s) & frame[target].notna()].dropna(subset=features)
        te = frame[(frame.season == s) & frame[target].notna()].dropna(subset=features)
        if len(te) < 60 or len(tr) < 200:
            continue
        m = HistGradientBoostingRegressor(
            max_iter=200, learning_rate=0.06, max_leaf_nodes=15, min_samples_leaf=40, random_state=0
        ).fit(tr[features], tr[target])
        rows.append(
            pd.Series({"n": len(te), "r": np.corrcoef(m.predict(te[features]), te[target])[0, 1]})
        )
    d = pd.DataFrame(rows)
    if d.empty:
        return (float("nan"), 0)
    return round(float(np.average(d.r, weights=d.n)), 3), int(d.n.sum())


print("\n(1) predicting NEXT-SEASON OUTPUT (npxG+xA per 90), held out by season, per role:")
for role in ["CB", "FB", "CM", "W", "ST"]:
    g = XP[XP.role == role]
    narrow = lfo_r(g, ["output", "npxg", "xa"], "output_next")
    wide = lfo_r(g, FEATS, "output_next")
    print(f"  {role}: output history alone r={narrow[0]} | full profile r={wide[0]}  (n={wide[1]})")

21447 player-seasons | with work-rate: 91.2% | with a next season: 72.8%

(1) predicting NEXT-SEASON OUTPUT (npxG+xA per 90), held out by season, per role:


  CB: output history alone r=0.185 | full profile r=0.024  (n=278)


  FB: output history alone r=0.456 | full profile r=0.446  (n=250)


  CM: output history alone r=0.619 | full profile r=0.605  (n=278)


  W: output history alone r=0.521 | full profile r=0.538  (n=293)


  ST: output history alone r=0.464 | full profile r=nan  (n=0)


In [37]:
print(
    "(2) predicting NEXT-SEASON MINUTES, beyond minutes+age (does the profile see quality managers reward?):"
)
players_dob = tm_loader.load_table("players")[["player_id", "date_of_birth"]]
players_dob["tm_player_id"] = players_dob.player_id.astype(str)
XP2 = XP.merge(players_dob[["tm_player_id", "date_of_birth"]], on="tm_player_id", how="left")
XP2["age"] = XP2.season + 1 - pd.to_datetime(XP2.date_of_birth).dt.year
XP2["minutes_next_filled"] = XP2.minutes_next.fillna(0)
for role in ["CB", "FB", "CM", "W", "ST"]:
    g = XP2[XP2.role == role].dropna(subset=["age"])
    narrow = lfo_r(g, ["minutes", "age"], "minutes_next_filled")
    wide = lfo_r(g, ["minutes", "age"] + FEATS, "minutes_next_filled")
    print(f"  {role}: minutes+age r={narrow[0]} | + full profile r={wide[0]}  (n={wide[1]})")

print("\n(3) predicting NEXT-JULY MARKET VALUE, beyond the market model's own inputs:")
import json

mrows = pd.DataFrame(json.load(open(config.MODELS / "phase3_market.json"))["rows"])
XP3 = XP2.merge(
    mrows[["player_id", "season", "value_next_july", "value_july"]].drop_duplicates(
        ["player_id", "season"]
    ),
    on=["player_id", "season"],
    how="inner",
)
XP3 = XP3[XP3.value_next_july.notna() & XP3.value_july.notna()].copy()
XP3["y"] = np.log10(XP3.value_next_july)
XP3["log_prior"] = np.log10(XP3.value_july)
for role in ["CB", "FB", "CM", "W", "ST"]:
    g = XP3[XP3.role == role].dropna(subset=["age"])
    narrow = lfo_r(g, ["output", "log_prior", "minutes", "age"], "y")
    wide = lfo_r(g, ["output", "log_prior", "minutes", "age"] + FEATS, "y")
    print(
        f"  {role}: market-model inputs r={narrow[0]} | + full profile r={wide[0]}  (n={wide[1]})"
    )

(2) predicting NEXT-SEASON MINUTES, beyond minutes+age (does the profile see quality managers reward?):


  CB: minutes+age r=0.406 | + full profile r=0.423  (n=370)


  FB: minutes+age r=0.34 | + full profile r=0.278  (n=307)


  CM: minutes+age r=0.422 | + full profile r=0.296  (n=371)


  W: minutes+age r=0.392 | + full profile r=0.431  (n=372)


...

  CB: market-model inputs r=0.928 | + full profile r=0.917  (n=300)


  FB: market-model inputs r=0.922 | + full profile r=0.903  (n=262)


  CM: market-model inputs r=0.924 | + full profile r=0.901  (n=316)


  W: market-model inputs r=0.928 | + full profile r=0.914  (n=319)


  ST: market-model inputs r=0.927 | + full profile r=nan  (n=0)


### Step 10 — what we got

**The wider profile earns no weights, against any of the three outcomes.** Per role, held out by
season, gradient boosting on the full validated stat list (six output quantities + ten work-rate
metrics) versus the narrow inputs:

| outcome | narrow inputs | + full profile |
|---|---|---|
| next-season output (npxG+xA/90) | CB 0.185 · FB 0.456 · CM 0.619 · W 0.521 | CB 0.024 · FB 0.446 · CM 0.605 · W 0.538 |
| next-season minutes (beyond minutes + age) | CB 0.406 · FB 0.340 · CM 0.422 · W 0.392 · ST 0.421 | CB 0.423 · FB 0.278 · CM 0.296 · W 0.431 · ST 0.320 |
| next-July market value (beyond prior, output, minutes, age) | 0.922–0.928 in every role | 0.901–0.917 in every role |

- On future output the profile never wins (for centre-backs it collapses — sixteen features
  overfit ~2,000 training rows). On future minutes it helps two roles and hurts three, the
  signature of variance, not signal. On future price it is slightly worse everywhere: the market's
  valuation is already carried by last year's value, output, minutes and age.
- Strikers could not be scored in the first and third tests (one profile field lacks coverage for
  them and the strict feature requirement empties their test sets); the four other roles are
  consistent.
- **Decision:** no composite "role score" — with fitted weights or otherwise. The ranked number
  stays npxG + xA per 90 within role; the profile (ball-winning, build-up, shot locations, role
  shares) stays descriptive and drives eligibility and "players like X". This was run as the
  direct answer to the owner's objection ("why would Rodri rank below a striker?"): first, he
  never does — rankings are within role only; second, even the outcomes that plainly value
  Rodri-types (managers' minutes, the market's price) do not reward the wide profile beyond the
  obvious inputs at this sample size. Fourth tested-and-declined intuition for the writeup
  (after the big-game slope, outcome-based defensive value, and style fit).
- Safety net unchanged: Phase 5 reports backtest results **split per role**, so "does this work
  for defenders/midfielders?" gets its own number.